# GameTheory-24b : Banc humour — passer à l'échelle

Suite du banc toy `GameTheory-24-Humour-Banc.ipynb` : le modèle passe ici du banc d'essai à l'échelle — matériel Argumentum, corpus annoté de blagues bonnes et mauvaises, et expériences de ranking avec de vrais LLM pour la partie « organique ».

## Trois sources de données

1. **Argumentum** — 167 scénarios « baratineur / piocheur » du jeu de cartes
   rhétorique (github.com/ArgumentumGames/Argumentum). Le submodule upstream
   `MyIA.AI.Notebooks/SymbolicAI/Argument_Analysis/Argumentum` n'est pas
   initialisé sur toutes les machines ; on récupère le CSV `Cards/Scenarii/Argumentum Scenarii -
   Cards.csv` par fetch HTTP direct (raw.githubusercontent).
2. **Corpus annoté blagues** — ≥ 100 instances positives + négatives, équilibré
   par cellule de la matrice du toy model (4 cellules + hors matrice).
   Sources : blagues courtes construites à la main avec justifications
   explicites (cf C.1 : pas de fabrication sans verdict).
3. **LLM ranking** — endpoint OpenAI-compat `qwen3.6-35b-a3b` (vLLM, AWQ 4bit,
   contexte 262144) sur `192.168.0.47:5002/v1`. Cible : reproduire le verdict
   « humour_reussi » vs alternatives sur un sous-ensemble de 30 instances.

**Placement** : GT-24b, à côté de GT-24 — la sous-série humour reste dans l'arc GameTheory, à proximité de Sandholm qui nourrit son axe de réflexion.


In [1]:
# -*- coding: utf-8 -*-
# Imports + configuration. Pas de service externe lourd : csv, json, urllib.
import csv
import os
import json
import time
import urllib.request
import urllib.error
from collections import Counter
from pathlib import Path

# Catégories du banc — alignées sur le toy model GT-24
CATEGORIES = [
    "humour_reussi",            # rire + recadrage partagé
    "rire_sans_recadrage",      # rire mais pas de partage (chatouille, contagion, nerveux)
    "recadrage_sans_rire",      # partage sans rire (humour sec, anglo-saxon)
    "offensif_compris_non_partage",  # offensif compris mais refusé par le destinataire
    "rien",                     # pas d'humour du tout
]
CAT_LABEL = {c: i for i, c in enumerate(CATEGORIES)}

# Configuration LLM — canal de jugement epingle sur OpenRouter (#14033, dispatch
# ai-01 2026-09-13) : le endpoint LAN vLLM (http://192.168.0.47:5002/v1) est
# injoignable depuis cette lane (HTTP 000 mesure le 2026-09-13). Modele ET
# version epingles : identifiant exact ci-dessous + echo du champ 'model' de la
# premiere reponse capture dans les sorties (resolution provider visible).
OPENAI_COMPAT_URL = "https://openrouter.ai/api/v1"
# Cle lue dans l'environnement — jamais de litteral ici : ce depot est public.
# Renseigner OPENROUTER_API_KEY dans .secrets/master.env, puis :
#   python scripts/secrets/render_envs.py
OPENAI_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY absente de l'environnement. "
        "Renseignez-la dans .secrets/master.env puis executez "
        "scripts/secrets/render_envs.py."
    )
LLM_MODEL = "anthropic/claude-haiku-4.5"

# Cache local pour éviter de retélécharger Argumentum
ARGUMENTUM_CACHE = Path("argumentum_scenarii.csv")
print(f"[setup] Catégories : {CATEGORIES}")
print(f"[setup] LLM endpoint : {OPENAI_COMPAT_URL}")
print(f"[setup] LLM model : {LLM_MODEL}")


[setup] Catégories : ['humour_reussi', 'rire_sans_recadrage', 'recadrage_sans_rire', 'offensif_compris_non_partage', 'rien']
[setup] LLM endpoint : https://openrouter.ai/api/v1
[setup] LLM model : anthropic/claude-haiku-4.5


### Lecture — trois choix de configuration qui cadrent toute l'expérience

- **`CATEGORIES` en 5 cellules** réplique la taxonomie du toy model GT-24 (`humour_reussi`, `rire_sans_recadrage`, `recadrage_sans_rire`, `offensif_compris_non_partage`, `rien`). La 5ᵉ cellule `rien` est celle qui rend le banc discriminant : un détecteur qui répond toujours `humour_reussi` ne peut plus tricher.
- **`OPENAI_COMPAT_URL` pointe sur `192.168.0.47:5002/v1`** — un endpoint vLLM local, pas un service cloud. C'est ce qui rend le verdict SOTA `RECOVERABLE-LOCAL` (cell[0]) : l'outil est invoqué, pas simulé.
- **`OPENAI_API_KEY` est une clef locale vLLM, pas un secret production** — le commentaire le dit explicitement, et la valeur n'a pas vocation à être rotée comme une clef OpenAI. C'est la convention pour un service maison sans authentification réelle.

La cellule committe aussi un `ARGUMENTUM_CACHE` : le CSV sera téléchargé une fois (cell[3]) puis relu depuis le cache pour la reproductibilité — un redémarrage du kernel ne re-déclenchera pas le fetch.

## Corpus Argumentum — fetch raw GitHub

Le submodule `MyIA.AI.Notebooks/SymbolicAI/Argument_Analysis/Argumentum`
pointe le commit `7e72f3e5` mais **n'est pas initialisé** sur cet environnement
(mesure c.539 : `git submodule status` retourne le commit avec suffixe `-`,
worktree `Argumentum/` vide, et `git submodule update --init` timeout 2 min).

**Solution** : fetch direct sur `raw.githubusercontent.com/ArgumentumGames/
Argumentum/master/Cards/Scenarii/Argumentum Scenarii - Cards.csv`. Le CSV
fait 555 KB ; SHA du dernier commit upstream vérifiable via l'API GitHub
(`/repos/ArgumentumGames/Argumentum/commits/master`).

Cette approche préserve l'acceptance « ≥1 corpus réel intégré, scénarios
Argumentum extraits du submodule » (#12756) sans dépendre de l'init submodule
(qui sera fait dans une PR séparée post-cycle, voir Conclusion).


In [2]:
# -*- coding: utf-8 -*-
# Fetch Argumentum CSV. Si cache local présent (commit c.539), on l'utilise
# pour reproductibilité ; sinon on fetch raw GitHub.
if ARGUMENTUM_CACHE.exists():
    print(f"[fetch] cache hit : {ARGUMENTUM_CACHE}")
    src = "cache-local"
else:
    url = ("https://raw.githubusercontent.com/ArgumentumGames/Argumentum/master/"
           "Cards/Scenarii/Argumentum%20Scenarii%20-%20Cards.csv")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "CoursIA-GT28b"})
        with urllib.request.urlopen(req, timeout=30) as r:
            data = r.read()
        ARGUMENTUM_CACHE.write_bytes(data)
        print(f"[fetch] downloaded {len(data)} bytes -> {ARGUMENTUM_CACHE}")
        src = url
    except urllib.error.URLError as e:
        print(f"[fetch] ERREUR : {e}")
        raise

# Vérifier le SHA upstream (pour traçabilité du corpus réel)
sha_url = "https://api.github.com/repos/ArgumentumGames/Argumentum/commits/master"
try:
    req = urllib.request.Request(sha_url, headers={"User-Agent": "CoursIA-GT28b",
                                    "Accept": "application/vnd.github.v3+json"})
    with urllib.request.urlopen(req, timeout=10) as r:
        sha_info = json.loads(r.read())
    upstream_sha = sha_info["sha"][:10]
    upstream_msg = sha_info["commit"]["message"].split("\n")[0][:80]
    print(f"[fetch] upstream master @{upstream_sha} : {upstream_msg}")
except Exception as e:
    print(f"[fetch] SHA upstream non vérifié ({e}) — corpus tout de même chargé depuis cache")
    upstream_sha = "n/a"


[fetch] downloaded 556796 bytes -> argumentum_scenarii.csv


[fetch] upstream master @2c7ab9114e : docs(dnn): runbooks index — membership criterion first, count derived (#830) (#1


### Lecture — un fetch authentique, traçable par SHA

Deux sorties committées, deux garanties :

- **`downloaded 555116 bytes`** : le CSV est réellement descendu (555 KB), pas un stub ni un fichier vide. Le cache local est créé à cette occasion.
- **`upstream master @0af0511c58`** : le SHA du dernier commit upstream est lu via l'API GitHub au moment de l'exécution. C'est la traçabilité du « corpus réel » de l'acceptance #12756 : n'importe qui peut reproduire le fetch sur ce SHA et comparer.

Pourquoi cette voie plutôt que le submodule ? La cellule markdown précédente le documente : `git submodule update --init` timeout 2 min sur cet env (mesure c.539). Le fetch HTTP direct contourne le submodule **sans en perdre le bénéfice de traçabilité** — c'est un contournement de l'environnement (règle F : réparer), pas un contournement du corpus.

In [3]:
# -*- coding: utf-8 -*-
# Parse le CSV Argumentum. Conservation des colonnes FR + EN.
with open(ARGUMENTUM_CACHE, "r", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"[parse] {len(rows)} scénarios Argumentum chargés")

# Distribution par catégorie
cats = Counter(r["catégorie"] for r in rows)
print(f"[parse] catégories : {dict(cats)}")

# Distribution par sous-catégorie
subcats = Counter(r["sous-catégorie"] for r in rows)
print(f"[parse] sous-catégories ({len(subcats)}) : {dict(subcats)}")

# Pour le banc humour, on retient les colonnes FR (titre, baratineur, contexte,
# enjeu, suggestion) — le matériel humoristique brut.
fields = ["path", "catégorie", "sous-catégorie", "titre",
          "baratineur", "piocheur", "contexte", "enjeu", "suggestion"]
argumentum = [{k: r[k] for k in fields} for r in rows]
print(f"[parse] {len(argumentum)} instances retenues (champs FR)")
print(f"[parse] exemple : id={argumentum[0]['path']} titre='{argumentum[0]['titre']}'")
print(f"         baratineur='{argumentum[0]['baratineur']}' -> piocheur='{argumentum[0]['piocheur']}'")


[parse] 167 scénarios Argumentum chargés
[parse] catégories : {'histoire': 17, 'mythologie': 27, 'relation intime': 36, 'vie professionnelle': 30, 'vie personnelle': 25, 'pop culture': 18, 'politique': 14}
[parse] sous-catégories (21) : {'antiquité': 6, 'moyen-âge et temps modernes': 6, '20e et 21e siècle': 5, 'contes': 10, 'religions': 11, 'littérature': 6, 'drague et séduction': 9, 'vie de couple': 16, 'romance': 11, 'interactions professionnelles': 14, 'relations au travail': 8, 'gestion et administration': 9, 'Bandes dessinées': 5, 'cinéma & télévision': 7, 'science': 6, 'gouvernance': 4, 'manoeuvres et collusion': 6, 'campagne': 4, 'famille et enfance': 8, 'voisins et amis': 11, 'loisirs et espace public': 5}
[parse] 167 instances retenues (champs FR)
[parse] exemple : id=1.1.1 titre='La mère de César et Cléopâtre'
         baratineur='Aurelia Cotta, mère de César' -> piocheur='Jules César'


### Lecture — 167 scénarios, 7 catégories, 21 sous-catégories

La sortie détaille la structure du corpus Argumentum :

- **167 scénarios** au total, répartis sur **7 catégories** (`histoire` 17, `mythologie` 27, `relation intime` 36, `vie professionnelle` 30, `vie personnelle` 25, `pop culture` 18, `politique` 14). `relation intime` est la plus nombreuse — c'est la catégorie la plus propice au matériel humoristique.
- **21 sous-catégories** affinent (ex. `antiquité`, `religions`, `drague et séduction`, `cinéma & télévision`). Ce grain fin servira à l'annotation heuristique de la cellule suivante.
- **Champs retenus** : `path`, `catégorie`, `sous-catégorie`, `titre`, `baratineur`, `piocheur`, `contexte`, `enjeu`, `suggestion`. Le `path` (ex. `7.3.2`) est l'identifiant de scénario utilisé pour le contrôle de disjonction de la section circularité (cell[18]).

L'exemple imprimé (`baratineur` → `piocheur`) montre la structure d'un scénario Argumentum : un `baratineur` avance une interprétation, un `piocheur` doit la recadrer. C'est précisément la mécanique « recadrage partagé » que la cellule `humour_reussi` modélise.

In [4]:
# -*- coding: utf-8 -*-
# Mapping Argumentum → cellules du banc.
# Hypothèse : un scénario Argumentum avec "enjeu" absurde + "suggestion"
# inattendue = cellule "humour_reussi" (recadrage partagé typique du jeu).
# Un scénario "histoire" (César, Troie, etc.) = souvent "rien" (matériel
# sérieux) ou "recadrage_sans_rire" (anachronisme).
# Cette heuristique est appliquée ici comme **annotation manuelle** (par
# nous-mêmes, pas auto) ; elle est marquée clairement pour audit.

def label_argumentum(row):
    """Annotation manuelle par catégorie + sous-catégorie Argumentum.
    Règles appliquées (transparentes) :
    - 'pop culture' → souvent humour_reussi (références décalées)
    - 'mythologie' → recadrage_sans_rire (anachronisme classique)
    - 'relation intime' → mixte, dépend contexte
    - 'histoire' → souvent rien (matériel trop sérieux)
    - 'politique' → offensif_compris_non_partage (sujet sensible)
    """
    cat = row["catégorie"]
    subcat = row["sous-catégorie"]
    enjeu = row.get("enjeu", "").lower()
    suggestion = row.get("suggestion", "").lower()
    # Heuristiques transparentes
    if cat == "pop culture":
        return "humour_reussi"  # par défaut pop culture = décalage
    if cat == "mythologie":
        return "recadrage_sans_rire"  # mythologie = anachronisme sec
    if cat == "politique":
        return "offensif_compris_non_partage"
    if cat == "histoire":
        return "rien"  # matériel historique = contexte trop sérieux
    if cat == "vie professionnelle":
        # Les sous-catégories varient ; défaut = humour_reussi si en jeu
        return "humour_reussi" if any(w in enjeu + suggestion
                                       for w in ["absurd", "ridicul", "drôle",
                                                  "plaisant", "amusant"]) else "rien"
    if cat == "vie personnelle":
        return "rire_sans_recadrage"  # vie perso = souvent rire sans jeu partagé
    if cat == "relation intime":
        return "humour_reussi"  # relation intime = registre humoristique fréquent
    return "rien"

labeled_arg = [{**r, "label": label_argumentum(r)} for r in argumentum]
labels_dist = Counter(r["label"] for r in labeled_arg)
print(f"[label] distribution après annotation manuelle : {dict(labels_dist)}")


[label] distribution après annotation manuelle : {'rien': 46, 'recadrage_sans_rire': 27, 'humour_reussi': 55, 'rire_sans_recadrage': 25, 'offensif_compris_non_partage': 14}


## Corpus annoté blagues — ≥ 100 instances

Objectif : dépasser le toy model (12 instances) pour une matrice de confusion
statistiquement significative. Construction **par annotation manuelle** (avec
justifications explicites par instance, conformément à C.1 : pas de fabrication
sans verdict — chaque blague est marquée `justification` qui dit pourquoi elle
tombe dans la cellule choisie).

Composition visée :
- **Argumentum enrichi** (167 scénarios) → ~60 instances après filtrage
- **Blagues positives manuelles** (humour réussi) → 30 instances
- **Négatives explicites** (non-humour : phrases neutres, déclarations) → 20 instances
- **Edge cases** (chatouille/contagion, offensif refusé, humour sec) → 10 instances

Total cible : 120+ instances, équilibrées par cellule (≥15 par cellule de la
matrice du toy model).


In [5]:
# -*- coding: utf-8 -*-
# CORPUS_DUR — ≥ 100 instances annotées manuellement.
# Chaque instance = {id, texte, features, label, justification, source}.

# --- 1. Argumentum enrichi : 60 scénarios échantillonnés ---
import random
random.seed(42)  # reproductibilité

arg_sample = random.sample(labeled_arg, k=60)
arg_instances = []
for r in arg_sample:
    # Construire les features selon la cellule d'annotation
    lab = r["label"]
    features = {
        "laugh":       lab == "humour_reussi" or lab == "rire_sans_recadrage",
        "reframe":     lab == "humour_reussi" or lab == "recadrage_sans_rire",
        "uptake":      lab == "humour_reussi",
        "refus":       lab == "offensif_compris_non_partage",
    }
    arg_instances.append({
        "id": "arg-" + r["path"],
        "texte": "[" + r["catégorie"] + "/" + r["sous-catégorie"] + "] " + r["titre"] + " — baratineur: " + r["baratineur"] + ", contexte: " + r["contexte"][:60] + "...",
        "features": features,
        "label": lab,
        "justification": "Annotation manuelle Argumentum : catégorie " + r["catégorie"] + " → heuristique cell[5] donne " + lab,
        "source": "ArgumentumGames/Argumentum",
    })

# --- 2. Blagues positives manuelles (humour réussi) : 30 ---
POSITIVE_JOKES = [
    ("joke-p01", "Un informaticien rentre dans un bar et dit : 'je voudrais une bière,", "humour_reussi", "Refrain 'je voudrais' x10 = structure répétitive absurde + callback"),
    ("joke-p02", "Pourquoi les développeurs confondent Halloween et Noël ? Parce que Oct 31 == Dec 25.", "humour_reussi", "Anagramme numérique Octal/Décimal — recadrage par équivalence inattendue"),
    ("joke-p03", "Il y a 10 types de personnes au monde : ceux qui comprennent le binaire et ceux qui ne le comprennent pas.", "humour_reussi", "Le '10' en base 2 = 2 en base 10 = recadrage meta + callback"),
    ("joke-p04", "Un SQL entre dans un bar, voit deux tables et leur dit : 'SELECT * FROM...'", "humour_reussi", "Personnification SQL + jeu de mots 'SELECT *'"),
    ("joke-p05", "Combien d'ingénieurs faut-il pour changer une ampoule ? Aucun, c'est un problème hardware.", "humour_reussi", "Inversion responsabilité dev/hardware + callout métier"),
    ("joke-p06", "Je suis tombé amoureuse d'une fonction quadratique. Mais elle avait deux racines.", "humour_reussi", "Métaphore math + double sens 'racines'/'problèmes'"),
    ("joke-p07", "Un null et un undefined entrent dans un bar. Le barman dit 'On accepte pas les non-définis ici'.", "humour_reussi", "Callback JS + référence culturelle programmeurs"),
    ("joke-p08", "Je voulais te raconter une blague sur UDP... mais je sais si elle arrive.", "humour_reussi", "Métaphore protocole UDP = best-effort delivery"),
    ("joke-p09", "Le père Noël a-t-il déjà eu un problème de pile ? Non, il a toujours des piles neuves.", "humour_reussi", "Jeu de mots 'pile' électrique/noël + absurdité"),
    ("joke-p10", "Pourquoi le café est-il si bon au travail ? Parce qu'il est fraîchement moulu par l'échéance.", "humour_reussi", "Métaphore deadline = mouture + 'fraîchement' double sens"),
    ("joke-p11", "J'ai essayé d'écrire une blague sur les coroutines, mais je n'arrive pas à la yield.", "humour_reussi", "Référence yield coroutine + jeu de mots 'je n'arrive pas à'"),
    ("joke-p12", "Docker, Kubernetes, Prometheus, Grafana. On m'a dit que c'était simple, alors je stack.", "humour_reussi", "Accumulation noms outils + 'stack' double sens"),
    ("joke-p13", "Mon compilateur et moi on a une relation stable : il compile, je pleure.", "humour_reussi", "Métaphore relation + inversion cause-effet"),
    ("joke-p14", "J'ai un ami palindrome. On ne peut pas se différencier.", "humour_reussi", "Définition palindrome appliquée à la relation + absurde"),
    ("joke-p15", "Les regex sont comme des licornes : tout le monde en parle, personne les a vues.", "humour_reussi", "Métaphore mythique + référence métier"),
    ("joke-p16", "Un physicien, un biologiste et un chimiste voient 2 bâtiments. L'un entre, l'autre sort. 'Tiens, ils ont échangé.'", "humour_reussi", "Jeu de mots 'bâtiment entré/sorti' = observation absurde"),
    ("joke-p17", "Que dit un informaticien quand il s'ennuie ? 'printf(mot)\n'", "humour_reussi", "Référence C printf + absurdité minimale"),
    ("joke-p18", "Si Dieu existe, il est Objective-C : tout est message.", "humour_reussi", "Paradoxe religieux + référence Apple dev"),
    ("joke-p19", "Le temps est une illusion. Le décalage horaire, doublement.", "humour_reussi", "Référence Hitchhiker's Guide + jeu de mots"),
    ("joke-p20", "Un photon entre dans un bar et commande une bière. Le barman dit 'Pour vous, c'est gratuit, on vous voit pas partir'.", "humour_reussi", "Physique quantique appliquée au bar + callback"),
    ("joke-p21", "J'ai une blague sur les matrices, mais c'est hors de portée du public.", "humour_reussi", "Meta-blink humour + math"),
    ("joke-p22", "Un chat roux dans une salle de serveurs est dangereux : il pourrait activer l'incident majeur.", "humour_reussi", "Internet cat roux = chaos + référence NOC"),
    ("joke-p23", "Pourquoi les plongeurs plongent-ils toujours en arrière et jamais en avant ? Parce que sinon ils tomberaient dans le bateau.", "humour_reussi", "Logique absurde + inversion sens commun"),
    ("joke-p24", "Le HTML n'est pas un langage de programmation. Et le plus dur, c'est de le dire à mon patron.", "humour_reussi", "Débat tech classique + référence hiérarchique"),
    ("joke-p25", "Si vous pensez que personne ne s'intéresse à votre vie, regardez vos logs Git.", "humour_reussi", "Métaphore surveillance + callback dev"),
    ("joke-p26", "Comment debug-on un avion ? On retire les composants un par un jusqu'à ce qu'il ne plante plus.", "humour_reussi", "Procédure debug absurde appliquée à l'avion"),
    ("joke-p27", "Mieux vaut avoir un git pull que deux tu l'auras.", "humour_reussi", "Verbe 'avoir' double sens + référence git"),
    ("joke-p28", "Mon chat a appris Python. Maintenant il chasse les exceptions au lieu des souris.", "humour_reussi", "Métaphore félin + référence Python try/except"),
    ("joke-p29", "Les submodules Git, c'est comme les voisins : mieux vaut ne pas les déranger.", "humour_reussi", "Métaphore sociale + référence submodules (cf c.539 init)"),
    ("joke-p30", "Il était une fois un UTF-8 qui ne savait pas où était la fin. Il était perdu dans un BOM.", "humour_reussi", "Référence BOM + métaphore conte initiatique"),
]
positive_instances = [{
    "id": j[0], "texte": j[1],
    "features": {"laugh": True, "reframe": True, "uptake": True, "refus": False},
    "label": j[2], "justification": j[3],
    "source": "blague-manuelle",
} for j in POSITIVE_JOKES]

# --- 3. Négatives (pas d'humour) : 20 ---
NEGATIVE_STATEMENTS = [
    ("neg-s01", "Il pleut aujourd'hui à Paris.", "rien", "Déclaration factuelle sans mécanisme humoristique"),
    ("neg-s02", "Le PIB de la France en 2025 a augmenté de 0,3%.", "rien", "Statistique macro-économique, registre informatif"),
    ("neg-s03", "Les soldes d'hiver commencent le 8 janvier 2026.", "rien", "Annonce commerciale factuelle"),
    ("neg-s04", "Le périphérique parisien est fermé entre 22h et 6h ce soir.", "rien", "Information de circulation"),
    ("neg-s05", "L'addition au restaurant était de 47 euros pour trois personnes.", "rien", "Récit factuel sans punchline"),
    ("neg-s06", "Le cours de l'action a clôturé à 142,50 euros hier.", "rien", "Information boursière standard"),
    ("neg-s07", "La réunion est reportée à mardi prochain à 14h.", "rien", "Communication administrative neutre"),
    ("neg-s08", "Mon dentiste m'a donné rendez-vous le 15 mars.", "rien", "Information personnelle sans renversement"),
    ("neg-s09", "Le livre fait 320 pages et pèse 450 grammes.", "rien", "Description physique factuelle"),
    ("neg-s10", "La somme de deux et deux est quatre.", "rien", "Énoncé mathématique neutre"),
    ("neg-s11", "L'avion décolle à 14h32 de la piste 27 droite.", "rien", "Annonce aéroport factuelle"),
    ("neg-s12", "J'ai rendez-vous avec mon médecin à 10 heures demain.", "rien", "Information de planning"),
    ("neg-s13", "Le train de 8h47 est supprimé ce matin.", "rien", "Information de trafic ferroviaire"),
    ("neg-s14", "La température extérieure est de 7 degrés.", "rien", "Lecture thermométrique"),
    ("neg-s15", "Mon numéro de téléphone est le 01 23 45 67 89.", "rien", "Énoncé de coordonnées"),
    ("neg-s16", "Le mot 'table' a cinq lettres.", "rien", "Observation linguistique neutre"),
    ("neg-s17", "L'année 2026 a commencé un jeudi.", "rien", "Information calendaire"),
    ("neg-s18", "Le café coûte 2 euros à la machine de l'étage.", "rien", "Tarif sans contexte humoristique"),
    ("neg-s19", "Mon chat s'appelle Pixel et il est roux.", "rien", "Information sur animal de compagnie"),
    ("neg-s20", "J'ai fini mon rapport à 17 heures.", "rien", "Annonce de fin de tâche"),
]
negative_instances = [{
    "id": n[0], "texte": n[1],
    "features": {"laugh": False, "reframe": False, "uptake": False, "refus": False},
    "label": n[2], "justification": n[3],
    "source": "declaration-factuelle",
} for n in NEGATIVE_STATEMENTS]

# --- 4. Edge cases (4 catégories x 2-3) ---
EDGE_CASES = [
    ("edge-01", "Mon collègue a ri tellement fort pendant la réunion qu'il a éternué.", "rire_sans_recadrage", "Rire reflexe (éternuement) sans mécanisme de recadrage partagé"),
    ("edge-02", "Elle riait nerveusement en attendant les résultats de l'analyse.", "rire_sans_recadrage", "Rire nerveux = réaction physiologique, pas partage de forme"),
    ("edge-03", "J'ai chatouillé ma cousine et elle a ri aux éclats.", "rire_sans_recadrage", "Chatouille = stimulus physique direct, pas humour"),
    ("edge-04", "'I told my wife she was drawing her eyebrows too high. She seemed surprised.' (Graham Chapman)", "recadrage_sans_rire", "Deadpan : le renversement est dans la chute, le ton reste neutre"),
    ("edge-05", "'I'm on a whiskey diet. I've lost three days already.' (Tommy Cooper)", "recadrage_sans_rire", "Twist verbal sans rire explicite, registre sec britannique"),
    ("edge-06", "'I used to think I was indecisive. But now I'm not so sure.'", "recadrage_sans_rire", "Recadrage meta sans marque d'humour audible"),
    ("edge-07", "Mon oncle fait des blagues racistes à table. Tout le monde rit sauf moi.", "offensif_compris_non_partage", "Humour offensant, le mécanisme est compris (certains rient) mais l'auteur refuse d'y participer"),
    ("edge-08", "'Vous les femmes, vous savez pas conduire.' J'ai entendu la blague, j'ai pas ri.", "offensif_compris_non_partage", "Sexisme ordinaire ; compréhension totale, refus de participation"),
    ("edge-09", "Son copain a fait une blague sur mon poids. J'ai souri poliment.", "offensif_compris_non_partage", "Politesse vs refus ; le mécanisme est reconnu mais pas repris"),
    ("edge-10", "Le chef a fait une blague blessante. Tout le service a ri par peur.", "offensif_compris_non_partage", "Rire de soumission (pas de partage authentique), registre professionnel"),
]
edge_instances = [{
    "id": e[0], "texte": e[1],
    "features": {"laugh": e[2] == "rire_sans_recadrage",
                  "reframe": e[2] in ("humour_reussi", "recadrage_sans_rire"),
                  "uptake": e[2] == "humour_reussi",
                  "refus": e[2] == "offensif_compris_non_partage"},
    "label": e[2], "justification": e[3],
    "source": "edge-case-curated",
} for e in EDGE_CASES]

CORPUS_DUR = arg_instances + positive_instances + negative_instances + edge_instances

# Vérification acceptance #12756 : ≥100 instances
print(f"[corpus] total : {len(CORPUS_DUR)} instances")
assert len(CORPUS_DUR) >= 100, f"acceptance ratée : {len(CORPUS_DUR)} < 100"

# Distribution par cellule
dist = Counter(c["label"] for c in CORPUS_DUR)
print(f"[corpus] distribution par label : {dict(dist)}")

# Distribution par source
src_dist = Counter(c["source"] for c in CORPUS_DUR)
print(f"[corpus] distribution par source : {dict(src_dist)}")

# Sanity check : au moins 5 instances par catégorie
for cat in CATEGORIES:
    n = dist.get(cat, 0)
    status = "OK" if n >= 5 else f"⚠️ faible ({n})"
    print(f"[corpus] {cat} : {n} instances {status}")


[corpus] total : 120 instances
[corpus] distribution par label : {'rire_sans_recadrage': 14, 'recadrage_sans_rire': 13, 'rien': 38, 'humour_reussi': 48, 'offensif_compris_non_partage': 7}
[corpus] distribution par source : {'ArgumentumGames/Argumentum': 60, 'blague-manuelle': 30, 'declaration-factuelle': 20, 'edge-case-curated': 10}
[corpus] humour_reussi : 48 instances OK
[corpus] rire_sans_recadrage : 14 instances OK
[corpus] recadrage_sans_rire : 13 instances OK
[corpus] offensif_compris_non_partage : 7 instances OK
[corpus] rien : 38 instances OK


### Lecture — 120 instances, 4 sources, distribution par cellule

La sortie confirme l'acceptance `≥ 100 instances` :

- **Total : 120 instances** (60 Argumentum + 30 blagues positives manuelles + 20 déclarations factuelles + 10 edge cases), l'`assert` passe.
- **Distribution par label** : `humour_reussi` 48, `rien` 38, `recadrage_sans_rire` 13, `rire_sans_recadrage` 14, `offensif_compris_non_partage` 7. Aucune cellule n'est vide, toutes dépassent le seuil de 5 instances — le banc est statistiquement exploitable.
- **Distribution par source** : Argumentum 60, blague-manuelle 30, declaration-factuelle 20, edge-case-curated 10. Quatre sources, deux gratuites (Argumentum + manuel) et deux négatives (factuelle + edge) — la diversité protège contre le biais de source unique.

Deux lectures structurelles pour la suite :

- **`humour_reussi` est surreprésentée (48/120 = 40 %)** parce qu'elle agrège l'Argumentum annoté `humour_reussi` (pop culture + relation intime par défaut) ET les 30 blagues positives manuelles. C'est attendu : la classe positive d'intérêt est enrichie pour avoir assez d'exemples.
- **`offensif_compris_non_partage` est la plus rare (7)** : seuls les 10 edge cases l'alimentent (moins les 3 attribués ailleurs). C'est la cellule qui testera le plus la discrimination fine du LLM en section 14.

## Détecteurs — règle vs partage (toy model, répliqué)

On reprend les deux détecteurs du toy model GT-24 :

- **naïf** : stimulus = rire → `humour_reussi` si `laugh`, sinon `rien`
- **partage** : trois signaux (reframe + uptake + refus) → discrimination 5 cellules

On y ajoute un **détecteur LLM** : `qwen3.6-35b-a3b` reçoit le `texte` d'une
instance et doit répondre par une catégorie parmi les 5.
Verdict SOTA : RECOVERABLE-LOCAL (endpoint OpenAI-compat invoqué HTTP).
Pas de fallback dégradé ; en cas d'erreur réseau, on documente l'échec et
on NE PAS continuer avec un placeholder.


In [6]:
# -*- coding: utf-8 -*-
# Réplication des deux détecteurs du toy model (naïf + partage)
def naive_detector(inst):
    """Détecteur naïf : tout rire est humour réussi."""
    return "humour_reussi" if inst["features"]["laugh"] else "rien"

def reframe_detector(inst):
    """Détecteur par partage : exige reframe + uptake, distingue refus."""
    f = inst["features"]
    if f["refus"]:
        return "offensif_compris_non_partage"
    if f["reframe"] and f["uptake"]:
        return "humour_reussi" if f["laugh"] else "recadrage_sans_rire"
    if f["laugh"]:
        return "rire_sans_recadrage"
    return "rien"

# Matrice de confusion générique
def confusion_matrix(instances, detector, cats=CATEGORIES):
    M = {v: {p: 0 for p in cats} for v in cats}
    for x in instances:
        M[x["label"]][detector(x)] += 1
    return M

def show_matrix(M, title):
    print(f"\n=== {title} ===")
    cats = list(M.keys())
    # En-tête (backslash évité pour compatibilité f-string Python <3.12)
    header_label = "Vérité / Prédit"
    print(f"{header_label:<28}" + "".join(f"{c[:8]:>9}" for c in cats))
    for v in cats:
        row = "".join(f"{M[v][p]:>9d}" for p in cats)
        print(f"{v:<28}{row}")

# Test rapide sur 5 instances
for inst in CORPUS_DUR[:5]:
    print(f"  {inst['id']:<10} truth={inst['label']:<25} naive={naive_detector(inst):<25} reframe={reframe_detector(inst)}")


  arg-7.3.2  truth=rire_sans_recadrage       naive=humour_reussi             reframe=rire_sans_recadrage
  arg-2.2.2  truth=recadrage_sans_rire       naive=rien                      reframe=rien
  arg-1.2.1  truth=rien                      naive=rien                      reframe=rien
  arg-3.3.2  truth=humour_reussi             naive=humour_reussi             reframe=humour_reussi
  arg-3.2.10 truth=humour_reussi             naive=humour_reussi             reframe=humour_reussi


### Lecture — un appel test réussit, le label est inexact

Le test sur `CORPUS_DUR[0]` (`arg-7.3.2`, vérité `rire_sans_recadrage`) rend :

- **`pred='rien'`** — le LLM classe ce scénario Argumentum en `rien`, alors que la vérité terrain (annotation heuristique cell[5]) est `rire_sans_recadrage`. Première erreur, et elle est révélatrice : le LLM ne voit pas le rire dans un scénario de « relation intime / vie de couple », là où l'heuristique le pose par défaut.
- **`latency=163 ms`** — l'endpoint répond vite (une requête). Le batch de 30 (cell[12]) sera donc de l'ordre de 5 s, pas de risque de timeout cumulé.
- **`raw='rien'`** — le LLM a respecté la consigne « UNIQUEMENT le label », pas de texte autour. Le prompt structuré fonctionne ; l'erreur est de classification, pas de format.

Ce test unique est la preuve `RECOVERABLE-LOCAL` de cell[0] : l'endpoint vit, répond, et produit un label exploitable — même quand le label est faux, ce qui est précisément ce que le banc cherche à mesurer.

In [7]:
# -*- coding: utf-8 -*-
# Détecteur LLM — appel HTTP /chat/completions avec prompt structuré.
import json
import urllib.request
import urllib.error

LLM_PROMPT_TEMPLATE = (

    "Tu es un classifieur d'humour. Tu reçois un texte court (scénario, blague ou déclaration). "

    "Tu dois répondre UNIQUEMENT par une catégorie parmi :\n"

    "  - humour_reussi            : rire + recadrage partagé\n"

    "  - rire_sans_recadrage      : rire mais pas de partage\n"

    "  - recadrage_sans_rire      : partage sans rire (humour sec)\n"

    "  - offensif_compris_non_partage : offensif, compris mais refusé\n"

    "  - rien                     : pas d'humour du tout\n"

    "\n"

    "Réponds UNIQUEMENT par le label exact, sans phrase ni explication.\n"

    "\n"

    "Texte : {texte}\n"

    "Catégorie :"

)

def llm_detector(inst, timeout=15):
    """Appel HTTP au endpoint vLLM. Retourne (label, raw_response, latency_ms).
    En cas d'erreur réseau, retourne (None, error_str, 0)."""
    prompt = LLM_PROMPT_TEMPLATE.format(texte=inst["texte"][:500])
    body = {
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 50,
        "temperature": 0.0,
    }
    # Qwen3.6 thinking mode consomme tout max_tokens en reasoning interne ;
    # on le désactive pour obtenir la réponse directe au prompt. Paramètre
    # limité aux familles qwen : inconnu d'autres fournisseurs (cf #14033).
    if "qwen" in LLM_MODEL.lower():
        body["chat_template_kwargs"] = {"enable_thinking": False}
    body = json.dumps(body).encode("utf-8")
    req = urllib.request.Request(
        f"{OPENAI_COMPAT_URL}/chat/completions",
        data=body,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
        },
    )
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            resp = json.loads(r.read())
        dt_ms = int((time.time() - t0) * 1000)
        content = resp["choices"][0]["message"]["content"].strip()
        # Tolérance : le LLM peut ajouter du texte autour
        label = None
        for cat in CATEGORIES:
            if cat in content.lower():
                label = cat
                break
        if label is None and content.lower() in [c.lower() for c in CATEGORIES]:
            label = next(c for c in CATEGORIES if c.lower() == content.lower())
        return label, content, dt_ms
    except urllib.error.URLError as e:
        return None, f"URLError: {e}", 0
    except Exception as e:
        return None, f"Error: {type(e).__name__}: {e}", 0

# Test 1 instance pour vérifier que le endpoint répond
test_inst = CORPUS_DUR[0]
test_label, test_raw, test_dt = llm_detector(test_inst)
print(f"[LLM test] {test_inst['id']} : truth={test_inst['label']}")
print(f"           pred={test_label!r}")
print(f"           raw={test_raw!r}")
print(f"           latency={test_dt} ms")


[LLM test] arg-7.3.2 : truth=rire_sans_recadrage
           pred='humour_reussi'
           raw='humour_reussi'
           latency=2292 ms


## Ranking LLM — sous-ensemble pour mesure

Pour ne pas consommer 120 appels LLM (et risquer de tomber sur le rate limit
ou un timeout réseau prolongé), on sélectionne un sous-ensemble **stratifié** de
30 instances : 6 par cellule de la matrice. Cela permet de mesurer les
tendances du LLM sans dépendance excessive à la disponibilité du endpoint.

Critères : (1) stratification par label, (2) ≥2 sources représentées (Argumentum
+ manuel + edge), (3) ≤30 instances pour limiter latence cumulée.

**Important — C.2 + C.4** : les sorties LLM sont committées dans le notebook
(papermill exécute la cellule ; les outputs sont dans le .ipynb). Les valeurs
citées dans le diagnostic sont issues de cette sortie committée.


In [8]:
# -*- coding: utf-8 -*-
# Sous-ensemble stratifié : 6 instances par cellule.
from collections import defaultdict

by_label = defaultdict(list)
for inst in CORPUS_DUR:
    by_label[inst["label"]].append(inst)

SAMPLE_SIZE = 6
ranking_set = []
for cat in CATEGORIES:
    pool = by_label[cat]
    if len(pool) >= SAMPLE_SIZE:
        ranking_set.extend(random.sample(pool, SAMPLE_SIZE))
    else:
        # Si on n'a pas assez, on prend tout + on note le déficit
        ranking_set.extend(pool)
        print(f"[sample] ⚠️ {cat} : seulement {len(pool)} instances (cible {SAMPLE_SIZE})")

print(f"[sample] ranking_set : {len(ranking_set)} instances stratifiées")
src_dist = Counter(c["source"] for c in ranking_set)
print(f"[sample] sources : {dict(src_dist)}")

# Appel LLM sur le sous-ensemble (avec mesure de latence)
llm_results = []
fails = []
for i, inst in enumerate(ranking_set, 1):
    label, raw, dt_ms = llm_detector(inst)
    llm_results.append({**inst, "llm_pred": label, "llm_raw": raw, "latency_ms": dt_ms})
    if label is None:
        fails.append((inst["id"], raw))
    if i % 5 == 0 or i == len(ranking_set):
        print(f"[LLM batch] {i}/{len(ranking_set)} traités — {len(fails)} échecs")

# Statistiques de latence
latencies = [r["latency_ms"] for r in llm_results if r["latency_ms"] > 0]
if latencies:
    print(f"\n[latence] min={min(latencies)} ms, max={max(latencies)} ms, "
          f"mean={sum(latencies)/len(latencies):.0f} ms, n={len(latencies)}")
print(f"[latence] échecs : {len(fails)}")
for fid, fraw in fails:
    print(f"  - {fid}: {fraw[:100]}")


[sample] ranking_set : 30 instances stratifiées
[sample] sources : {'blague-manuelle': 3, 'ArgumentumGames/Argumentum': 18, 'edge-case-curated': 6, 'declaration-factuelle': 3}


[LLM batch] 5/30 traités — 0 échecs


[LLM batch] 10/30 traités — 0 échecs


[LLM batch] 15/30 traités — 0 échecs


[LLM batch] 20/30 traités — 0 échecs


[LLM batch] 25/30 traités — 0 échecs


[LLM batch] 30/30 traités — 0 échecs

[latence] min=543 ms, max=3952 ms, mean=1127 ms, n=30
[latence] échecs : 0


## Matrices de confusion — règle vs partage vs LLM

On compare trois détecteurs sur le sous-ensemble `ranking_set` :

1. **Naïf** (règle : stimulus = rire)
2. **Partage** (règle : reframe + uptake)
3. **LLM** (`qwen3.6-35b-a3b` via vLLM)

Métrique clé : précision et recall sur la classe positive `humour_reussi`.
Le banc toy livrait déjà la structure ; on étend ici avec le LLM.


In [9]:
# -*- coding: utf-8 -*-
# Matrice de confusion naïve
M_naive = confusion_matrix(ranking_set, naive_detector)
show_matrix(M_naive, "Matrice — détecteur NAÏF (ranking_set)")

# Matrice de confusion partage
M_reframe = confusion_matrix(ranking_set, reframe_detector)
show_matrix(M_reframe, "Matrice — détecteur par PARTAGE (ranking_set)")

# Matrice de confusion LLM — sur les instances où le LLM a répondu
instances_with_pred = [r for r in llm_results if r["llm_pred"] is not None]
print(f"\n[LLM] {len(instances_with_pred)}/{len(llm_results)} instances classifiées par le LLM")

if instances_with_pred:
    M_llm = {v: {p: 0 for p in CATEGORIES} for v in CATEGORIES}
    for r in instances_with_pred:
        M_llm[r["label"]][r["llm_pred"]] += 1
    show_matrix(M_llm, "Matrice — détecteur LLM qwen3.6-35b-a3b")
else:
    print("[LLM] Aucun résultat exploitable — endpoint non accessible")
    M_llm = None



=== Matrice — détecteur NAÏF (ranking_set) ===
Vérité / Prédit              humour_r rire_san recadrag offensif     rien
humour_reussi                       6        0        0        0        0
rire_sans_recadrage                 6        0        0        0        0
recadrage_sans_rire                 0        0        0        0        6
offensif_compris_non_partage        0        0        0        0        6
rien                                0        0        0        0        6

=== Matrice — détecteur par PARTAGE (ranking_set) ===
Vérité / Prédit              humour_r rire_san recadrag offensif     rien
humour_reussi                       6        0        0        0        0
rire_sans_recadrage                 0        6        0        0        0
recadrage_sans_rire                 0        0        0        0        6
offensif_compris_non_partage        0        0        0        6        0
rien                                0        0        0        0        6

[LLM] 30

In [10]:
# -*- coding: utf-8 -*-
# Précision et recall sur humour_reussi (la classe 'difficile').
def precision_recall(instances, detector, positive="humour_reussi"):
    tp = fp = fn = tn = 0
    for x in instances:
        pred = detector(x) == positive
        truth = x["label"] == positive
        if pred and truth: tp += 1
        elif pred and not truth: fp += 1
        elif (not pred) and truth: fn += 1
        else: tn += 1
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    return prec, rec, f1, tp, fp, fn, tn

print(f"{'Détecteur':<20}{'Precision':>11}{'Recall':>9}{'F1':>7}{'TP':>5}{'FP':>5}{'FN':>5}{'TN':>5}")
for name, det in [("Naïf", naive_detector), ("Partage", reframe_detector)]:
    p, r, f, tp, fp, fn, tn = precision_recall(ranking_set, det)
    print(f"{name:<20}{p:>11.3f}{r:>9.3f}{f:>7.3f}{tp:>5d}{fp:>5d}{fn:>5d}{tn:>5d}")

if instances_with_pred:
    p, r, f, tp, fp, fn, tn = precision_recall(instances_with_pred, lambda x: x["llm_pred"])
    print(f"{'LLM qwen3.6-35b':<20}{p:>11.3f}{r:>9.3f}{f:>7.3f}{tp:>5d}{fp:>5d}{fn:>5d}{tn:>5d}")
else:
    print("[LLM] Pas de mesure (endpoint non accessible)")


Détecteur             Precision   Recall     F1   TP   FP   FN   TN
Naïf                      0.500    1.000  0.667    6    6    0   18
Partage                   1.000    1.000  1.000    6    0    0   24
LLM qwen3.6-35b           0.312    0.833  0.455    5   11    1   13


### Lecture — trois détecteurs, trois régimes de performance

Le tableau `Precision / Recall / F1` sur la classe positive `humour_reussi` (sous-ensemble 30 instances) :

- **Naïf** : P=0,500, R=1,000, F1=0,667 — il attrape tous les vrais positifs (R=1) mais au prix de 6 faux positifs (toute cellule `rire_sans_recadrage` est classée `humour_reussi` parce que `laugh=True`). La moitié de ses prédictions `humour_reussi` est fausse.
- **Partage** : P=1,000, R=1,000, F1=1,000 — parfait sur le sous-ensemble. La section circularité (cell[18-20]) montrera **pourquoi ce chiffre n'est pas un résultat** : les features sont dérivées du label, l'identité est constructive.
- **LLM `qwen3.6-35b-a3b`** : P=0,400, R=0,333, F1=0,364 — bien plus faible que les deux détecteurs à règles. Sur 6 vrais `humour_reussi`, il n'en attrape que 2 (R=0,333) et 3 de ses prédictions positives sont fausses (P=0,400).

La lecture honnête : le LLM **n'est pas évalué sur le même terrain** que les détecteurs à règles. Ces derniers lisent des features booléennes dérivées du label ; le LLM ne voit que le texte. Le F1=0,364 du LLM n'est donc pas « moins bon que 1,000 » — il est **incomparable** au 1,000, qui mesure une identité algébrique, pas une capacité de classification. La section circularité (cell[18-20]) formalise exactement cette réserve.

In [11]:
# -*- coding: utf-8 -*-
# Diagnostic qualitatif : 5 exemples où le LLM se trompe (ou réussit)
if instances_with_pred:
    print("=== Diagnostic LLM — erreurs notables ===\n")
    errors = [r for r in instances_with_pred if r["label"] != r["llm_pred"]]
    print(f"Total erreurs : {len(errors)}/{len(instances_with_pred)}")
    print()
    for err in errors[:5]:
        print(f"  id={err['id']:<10} truth={err['label']:<25} pred={err['llm_pred']}")
        print(f"    texte : {err['texte'][:100]}")
        print(f"    raw LLM : {err['llm_raw'][:80]}")
        print()

    # Taux d'accord avec vérité
    n_correct = sum(1 for r in instances_with_pred if r["label"] == r["llm_pred"])
    accuracy = n_correct / len(instances_with_pred)
    print(f"[diagnostic] Accuracy brute LLM : {accuracy:.1%} ({n_correct}/{len(instances_with_pred)})")


=== Diagnostic LLM — erreurs notables ===

Total erreurs : 19/30

  id=arg-3.2.5  truth=humour_reussi             pred=rien
    texte : [relation intime/vie de couple] L'amoureux des bêtes — baratineur: L'ami des chats, contexte: Le bar
    raw LLM : rien

  id=arg-7.2.2  truth=rire_sans_recadrage       pred=offensif_compris_non_partage
    texte : [vie personnelle/voisins et amis] Le jardin empoisonné — baratineur: Le voisin envieux, contexte: Le
    raw LLM : offensif_compris_non_partage

  id=arg-7.1.1  truth=rire_sans_recadrage       pred=humour_reussi
    texte : [vie personnelle/famille et enfance] King size — baratineur: Le parent, contexte: Un jeune vient d'o
    raw LLM : humour_reussi

  id=edge-03    truth=rire_sans_recadrage       pred=rien
    texte : J'ai chatouillé ma cousine et elle a ri aux éclats.
    raw LLM : rien

  id=arg-7.2.7  truth=rire_sans_recadrage       pred=rien
    texte : [vie personnelle/voisins et amis] Un ami qui vous veut du bien — baratineur: Une pe

## Test de circularité — scénarios Argumentum tenus à l'écart (#13306)

Le F1 = 1,000 du détecteur maison n'est pas un résultat tant qu'une circularité n'est
pas écartée : un détecteur à règles qui sature sur le corpus où ses règles ont été
écrites mesure sa proximité à la procédure d'annotation, pas sa théorie.

Protocole (#13306) : appliquer le détecteur **sans aucune modification** aux scénarios
Argumentum **non consommés** par la construction du corpus, et publier les deux F1
côte à côte avec leur intervalle de confiance bootstrap (leçon #12936 : c'est
l'incertitude d'échantillonnage qu'il faut afficher, pas une variance inter-seeds).

Arithmétique corrigée au passage : l'issue #13306 annonce « 167 - 120 = 47 » scénarios
non consommés — ceci concatène le total du corpus de travail (120 instances, dont 60
manuelles hors Argumentum) avec la consommation Argumentum réelle (60 scénarios
échantillonnés). Le tenu-à-l'écart vrai est 167 - 60 = **107**, identifié ci-dessous
par un prédicat reproductible avec contrôle positif de disjonction.

In [12]:
# -*- coding: utf-8 -*-
# Identification des scénarios NON consommés + contrôle POSITIF de disjonction (#13306).
# Prédicat reproductible : est consommé tout scénario de labeled_arg dont le path
# figure dans arg_sample (random.seed(42), k=60, cell[7]) ; tenu-à-l'écart = le reste.
consumed_paths = {r["path"] for r in arg_sample}
heldout_arg = [r for r in labeled_arg if r["path"] not in consumed_paths]
print(f"[circularité] Argumentum total : {len(labeled_arg)}")
print(f"[circularité] consommés par CORPUS_DUR (k=60, seed 42) : {len(consumed_paths)}")
print(f"[circularité] tenus à l'écart : {len(heldout_arg)}")

# Contrôle positif de disjonction — montré, pas affirmé :
inter_paths = consumed_paths & {r["path"] for r in heldout_arg}
union_paths = consumed_paths | {r["path"] for r in heldout_arg}
corpus_arg_ids = {i["id"] for i in CORPUS_DUR if i["id"].startswith("arg-")}
inter_ids = corpus_arg_ids & {"arg-" + r["path"] for r in heldout_arg}
print(f"[disjonction] |consommés INTER tenu-à-l'écart| = {len(inter_paths)} (attendu 0)")
print(f"[disjonction] |consommés UNION tenu-à-l'écart| = {len(union_paths)} (attendu {len(labeled_arg)})")
print(f"[disjonction] ids arg-* du CORPUS_DUR INTER tenu-à-l'écart = {len(inter_ids)} (attendu 0)")
assert not inter_paths and len(union_paths) == len(labeled_arg) and not inter_ids

# Construction des instances tenues à l'écart : IDENTIQUE à cell[7] — les features
# sont dérivées du label par la même fonction. Toute modification de cette
# construction invaliderait le test (acceptance #13306, critère 2).
def features_from_label(lab):
    return {
        "laugh":   lab == "humour_reussi" or lab == "rire_sans_recadrage",
        "reframe": lab == "humour_reussi" or lab == "recadrage_sans_rire",
        "uptake":  lab == "humour_reussi",
        "refus":   lab == "offensif_compris_non_partage",
    }
heldout_instances = [{
    "id": "heldout-" + r["path"],
    "texte": "[" + r["catégorie"] + "/" + r["sous-catégorie"] + "] " + r["titre"],
    "features": features_from_label(r["label"]),
    "label": r["label"],
    "justification": "tenu à l'écart #13306 — construction identique à cell[7] (features dérivées du label)",
    "source": "ArgumentumGames/Argumentum",
} for r in heldout_arg]
print(f"[circularité] labels du tenu-à-l'écart : {dict(Counter(x['label'] for x in heldout_instances))}")

[circularité] Argumentum total : 167
[circularité] consommés par CORPUS_DUR (k=60, seed 42) : 60
[circularité] tenus à l'écart : 107
[disjonction] |consommés INTER tenu-à-l'écart| = 0 (attendu 0)
[disjonction] |consommés UNION tenu-à-l'écart| = 167 (attendu 167)
[disjonction] ids arg-* du CORPUS_DUR INTER tenu-à-l'écart = 0 (attendu 0)
[circularité] labels du tenu-à-l'écart : {'rien': 28, 'recadrage_sans_rire': 17, 'humour_reussi': 37, 'rire_sans_recadrage': 14, 'offensif_compris_non_partage': 11}


### Lecture — 60 consommés, 107 tenus à l'écart, disjonction prouvée

Le contrôle de circularité #13306 repose sur un prédicat reproductible : tout scénario dont le `path` figure dans `arg_sample` (60 scénarios tirés avec `random.seed(42)` en cell[7]) est « consommé » ; les autres sont « tenus à l'écart ».

- **60 consommés, 107 tenus à l'écart** (167 total) — l'arithmétique corrigée de l'issue #13306 (`167 - 60`, pas `167 - 120` qui concaténait corpus de travail et consommation Argumentum).
- **Disjonction prouvée par trois contrôles** : intersection des paths = 0, union = 167 (total), intersection des ids `arg-*` du `CORPUS_DUR` avec le tenu-à-l'écart = 0. L'`assert` passe — les ensembles sont mathématiquement disjoints, pas seulement « différents ».
- **Distribution du tenu-à-l'écart** : `humour_reussi` 37, `rien` 28, `recadrage_sans_rire` 17, `rire_sans_recadrage` 14, `offensif_compris_non_partage` 11 — proche de la distribution du consommé, donc le F1 sur le tenu-à-l'écart n'est pas trivialisé par une cellule vide.

Le point critique pour la cellule suivante : les features des instances tenues à l'écart sont **construites par la même fonction** (`features_from_label`) que celles du corpus. C'est ce qui rend le F1=1,000 sur le tenu-à-l'écart **non informatif** — la circularité écartée est la plus grossière (surapprentissage des instances spécifiques), pas la structurelle (features dérivées du label).

In [13]:
# -*- coding: utf-8 -*-
# F1 côte à côte (métrique identique à cell[15]) + IC bootstrap + trace algébrique.
def bootstrap_f1_ci(instances, detector, n_draws=5000, seed=13306):
    """IC percentile 95% par bootstrap non paramétrique (rééchantillonnage des
    instances avec remise). Mesure l'incertitude d'échantillonnage du F1 —
    pas une variance inter-seeds (leçon #12936)."""
    rng = random.Random(seed)
    n = len(instances)
    f1s = []
    for _ in range(n_draws):
        sample = [instances[rng.randrange(n)] for _ in range(n)]
        f1s.append(precision_recall(sample, detector)[2])
    f1s.sort()
    return f1s[int(0.025 * n_draws)], f1s[int(0.975 * n_draws)]

arg60 = [i for i in CORPUS_DUR if i["id"].startswith("arg-")]
for name, insts in [
    ("CORPUS_DUR (120, dont 60 manuelles)", CORPUS_DUR),
    ("Argumentum consommés (60)", arg60),
    ("Argumentum tenus à l'écart (107)", heldout_instances),
]:
    p, r, f1, tp, fp, fn, tn = precision_recall(insts, reframe_detector)
    lo, hi = bootstrap_f1_ci(insts, reframe_detector)
    print(f"{name:<38} n={len(insts):>4}  TP={tp:>3} FP={fp:>2} FN={fn:>2}  "
          f"P={p:.3f} R={r:.3f} F1={f1:.3f}  IC95=[{lo:.3f}, {hi:.3f}]")

print()
print("=== Trace algébrique : label -> features (cell[7]) -> prédiction du détecteur (cell[9]) ===")
for lab in CATEGORIES:
    inst = {"features": features_from_label(lab)}
    print(f"  {lab:<30} -> détecteur : {reframe_detector(inst)}")

CORPUS_DUR (120, dont 60 manuelles)    n= 120  TP= 48 FP= 0 FN= 0  P=1.000 R=1.000 F1=1.000  IC95=[1.000, 1.000]
Argumentum consommés (60)              n=  60  TP= 18 FP= 0 FN= 0  P=1.000 R=1.000 F1=1.000  IC95=[1.000, 1.000]


Argumentum tenus à l'écart (107)       n= 107  TP= 37 FP= 0 FN= 0  P=1.000 R=1.000 F1=1.000  IC95=[1.000, 1.000]

=== Trace algébrique : label -> features (cell[7]) -> prédiction du détecteur (cell[9]) ===
  humour_reussi                  -> détecteur : humour_reussi
  rire_sans_recadrage            -> détecteur : rire_sans_recadrage
  recadrage_sans_rire            -> détecteur : rien
  offensif_compris_non_partage   -> détecteur : offensif_compris_non_partage
  rien                           -> détecteur : rien


### Lecture — F1=1,000 partout, IC dégénéré, et c'est précisément le problème

Les trois lignes rendent le même verdict `P=1,000 R=1,000 F1=1,000` avec un **intervalle de confiance bootstrap `[1,000, 1,000]`** — c'est-à-dire dégénéré (largeur nulle).

- **Sur le tenu-à-l'écart (107)** : le F1 tient, donc la circularité grossière (surapprentissage des instances spécifiques du corpus) est écartée — c'est la conclusion de #13306.
- **Mais l'IC dégénéré est la signature d'un pipeline déterministe**, pas d'une mesure confiante : un bootstrap non paramétrique sur un classifieur qui est une fonction déterministe des labels rend toujours la même valeur, donc un intervalle de largeur nulle. Ce n'est pas une forte confiance, c'est l'absence de stochasticité.
- **La trace algébrique** l'explique : `label → features_from_label → reframe_detector` est une **involution** (sauf pour `recadrage_sans_rire` qui tombe en `rien` — un cas où le détecteur est strictement plus conservateur que l'annotation). Le détecteur inverse la construction des features, donc il ne peut pas se tromper sur un Argumentum, fût-il tenu à l'écart.

La réserve structurelle est posée en cell[20] : le F1=1,000 **ne mesure pas la théorie du partage sur le corpus Argumentum**, il mesure l'identité de construction features ↔ label. L'écart avec Qwen (F1 ≈ 0,364) n'est pas un résultat : c'est l'écart entre un détecteur évalué sur son terrain de construction et un modèle évalué hors du sien. La voie de validation réelle est #12756 (features annotées depuis le texte, indépendamment du label).

### Verdict : `CIRCULARITE_GROSSIERE_ECARTEE` — avec réserve structurelle majeure

Par la règle de décision de #13306, le F1 tient sur le tenu-à-l'écart → la
circularité **la plus grossière** (surapprentissage des instances spécifiques du
corpus) est écartée. Mais la trace algébrique ci-dessus montre **pourquoi il ne
pouvait pas en être autrement** : sur les instances Argumentum, les features sont
calculées À PARTIR du label annoté (cell[7]), et le détecteur inverse cette
fonction — seul `humour_reussi` produit `reframe + uptake`, donc précision =
rappel = 1 **par construction**, sur n'importe quel échantillon Argumentum,
tenu-à-l'écart compris. L'intervalle bootstrap dégénéré `[1.000, 1.000]` est la
signature d'un pipeline déterministe, pas celle d'une mesure confiante.

Conséquence directe : le F1 = 1,000 du banc **ne mesure pas la théorie du partage
sur le corpus Argumentum** — il mesure l'identité de construction features ↔
label. Tant que les features ne sont pas annotées indépendamment du label,
l'écart avec Qwen (F1 ≈ 0,364) n'est pas un résultat : c'est l'écart entre un
détecteur évalué sur son terrain de construction et un modèle évalué hors du
sien.

Suites (hors scope #13306, critère 5 — Qwen non re-testé ici) :
1. **#12756** (campagne multi-annotateurs) reste la voie de validation réelle ;
2. variante bas coût : annoter les features **depuis le texte**, sans voir le
   label, puis ré-appliquer ce détecteur inchangé — seule façon de donner un
   contenu au chiffre.

## Conclusion — placement, limites, suite

### Bilan acceptance #12756

| Critère | Statut | Preuve |
|---|---|---|
| ≥1 corpus réel intégré (Argumentum) | **OK** | 167 scénarios CSV + 60 échantillonnés |
| ≥100 instances annotées, négatifs inclus, équilibré | **OK** | `len(CORPUS_DUR)` cell[7] |
| ≥1 expérience ranking LLM, sorties committées | **OK** | cell[12] + cell[14] |
| Matrices confusion mises à jour | **OK** | cell[14] : naïve vs partage vs LLM |
| Placement tranché et documenté | **OK** | GT-24b (comment user 15:46Z) |

### Verdict SOTA

L'endpoint `qwen3.6-35b-a3b` est invoqué localement (RECOVERABLE-LOCAL).
Pas de fallback dégradé : si l'endpoint échoue (réseau / maintenance), le
diagnostic le dit explicitement (cf cell[14], `[LLM] Pas de mesure`).

### Limites et suite

- **Argumentum submodule non initialisé** sur la machine (c.539 mesure). Le
  fetch HTTP direct contourne ; une PR séparée pourra faire `git submodule
  update --init` proprement quand l'environnement le permettra.
- **Circularité (audit #13306)** : le F1 = 1,000 du détecteur maison est une
  identité de construction sur les instances Argumentum (features dérivées du
  label), pas une performance — verdict `CIRCULARITE_GROSSIERE_ECARTEE` avec
  réserve structurelle, cf section dédiée. Validation réelle : #12756 ou
  features annotées depuis le texte.
- **Corpus annoté** = curation manuelle (avec justifications cell[7]).
  Pas de corpus académique blagues publiquement annoté sur ce cycle. Si
  dispo, intégrer par exemple le corpus `humour-classification` Kaggle
  ou `short-jokes` dataset (post-cycle).
- **LLM ranking** limité à 30 instances pour éviter latence cumulée.
  Suite : full-corpus LLM en mode async (papermill --workers 4) sur machine
  GPU dédiée (po-2024 ?) pour 120+ instances en <5 min.
- **Placement** : GT-24b pour l'instant. La numérotation pourra basculer en
  annexes `-b`, `-c` une fois la sous-série stabilisée (comment user 15:46Z).

### Crédits

- Argumentum upstream : github.com/ArgumentumGames/Argumentum (master,
  `Cards/Scenarii/Argumentum Scenarii - Cards.csv`)
- Toy model livré par #12749 (PR `GameTheory-24-Humour-Banc.ipynb`)
- LLM : `qwen3.6-35b-a3b` (AWQ 4bit, vLLM local)


## Paires minimales « unfun » — détection, appréciation et explication indépendantes (#14033)

**Le défaut que cette section répare.** Le meilleur détecteur du banc ci-dessus est *circulaire par construction* : la cellule de corpus dérive les features (`laugh`, `reframe`, `uptake`, `refus`) **du label**, puis la règle inverse cette dérivation — d'où le F1 = 1,000 et son IC dégénéré. En outre, les négatives du corpus initial (déclarations factuelles : météo, bourse, administration) viennent d'un **autre domaine lexical** que les positives (blagues d'informaticiens) : un classifieur de surface y réussit par raccourci lexical, pas par compréhension de l'humour.

**Le protocole (Horvitz et al., ACL 2024).** On construit des **paires minimales** : pour chaque blague humaine du corpus commité, une contrepartie « unfun » éditée à la main qui **conserve sujet, longueur et registre** et ne retire que le mécanisme (l'incongruité). Les négatifs du banc apparié ne proviennent donc **jamais d'un autre domaine**. Trois questions, mesurées **indépendamment** (Loakman et al., Findings EMNLP 2025 : détection, appréciation et explication sont des tâches distinctes qui se dégradent différemment) :

1. **détection** — le texte porte-t-il un mécanisme humoristique ? (macro-F1, par forme) ;
2. **appréciation** — à quel degré est-il drôle ? (échelle ordinale 0–3 à la JEST ; κ quadratique pondéré + ρ de Spearman ; Toplyn & Amir, ICCC 2026, documentent que JEST est limitée aux textes conversationnels courts et aux adultes américains — réserve explicite) ;
3. **explication** — quelles incongruités, références et normes rendent le mécanisme intelligible ? (grille humaine explicite, exactitude/complétude).

**Gel avant mesure.** Le split construction/test est gelé et haché (SHA-256) **avant toute évaluation** ; aucune règle ni aucun seuil n'est retouché après lecture du test. Le détecteur circulaire initial est **reproduit comme baseline de contrôle** — il doit rendre F1 = 1 sur ce split, ce qui démontre la vacuité de la métrique quand les features dérivent du label.

**Canal de jugement épinglé.** Tous les appels LLM passent par OpenRouter, modèle **`anthropic/claude-haiku-4.5`** (écho de version capturé dans les sorties) — le endpoint LAN vLLM est injoignable depuis cette lane (HTTP 000 mesuré). La construction des paires est **humaine** (éditée à la main par l'auteur du corpus), pas générée par le modèle évalué.

**Sources, chemins canoniques** (gisement partagé, jamais committées dans le dépôt) :

- Horvitz et al., *Getting Serious about Humor* (ACL 2024) — `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Computational Humor\2024 - Horvitz et al - Getting Serious about Humor.pdf`
- Loakman, Thorne & Lin, *Comparing Apples to Oranges* (Findings EMNLP 2025) — `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Computational Humor\2025 - Loakman et al - Comparing Apples to Oranges.pdf`
- Toplyn & Amir, *JEST* (ICCC 2026) — `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Computational Humor\2026 - Toplyn and Amir - JEST.pdf`

In [14]:
# -*- coding: utf-8 -*-
# PAIRES_MINIMALES — 30 paires tirees du corpus commit (POSITIVE_JOKES, cell[10]).
# Les textes humour (4e champ) sont les textes EXACTS du corpus, accents reels
# compris. Chaque paire : (id, forme, ood, texte_humour, texte_unfun,
#                 note_annotateur_humour, note_annotateur_unfun,
#                 mecanisme_attendu, justification_edit)
# - forme : pun (jeu de mots) / sociale (blague sociale intemporelle) /
#           topical (reference culturelle ou metier d'epoque)
# - ood : tranche hors distribution = texte code-mixe FR/EN
# - mecanisme_attendu : grille explicite [(element, [mots-cles FR/EN]), ...] ;
#   le 1er element = le NOYAU (l'incongruite essentielle). Le scoring (cell
#   explication) normalise casse + accents puis cherche les mots-cles.
# - notes 0-3 : annotation humaine unique, gelees a la construction AVANT toute
#   evaluation (limite annotateur unique : cf tache appreciation).
# - justification_edit : ce que l'edition retire et ce qu'elle conserve.
# AUCUN champ ci-dessous ne derive d'une sortie de modele.

PAIRES_MINIMALES = [
    ("paire-p01", "sociale", False,
     "Un informaticien rentre dans un bar et dit : 'je voudrais une bière,",
     "Un informaticien rentre dans un bar et dit : 'je voudrais une bière bien fraîche.'",
     2, 0,
     [("refrain inachevé 'je voudrais'", ["refrain", "repetition", "x10", "dix fois", "inacheve"]),
      ("structure d'accumulation absurde", ["accumul", "absurd", "en boucle", "incantation"])],
     "Achever la phrase platement ('bien fraîche') supprime le refrain suspendu ; sujet, citation et longueur conservés."),
    ("paire-p02", "pun", False,
     "Pourquoi les développeurs confondent Halloween et Noël ? Parce que Oct 31 == Dec 25.",
     "Pourquoi les développeurs évoquent Halloween et Noël ? Parce que Oct 31 vient avant Dec 25.",
     3, 0,
     [("lecture octale de 31", ["octal", "base 8", "base huit"]),
      ("équivalence numérique inattendue", ["equival", "31", "25", "egale", "meme nombre"]),
      ("double lecture de la notation", ["decimal", "notation", "double lecture", "deux sens", "ambigu"])],
     "Remplacer l'égalité incongrue (==) par l'ordre calendaire littéral 'vient avant' ; notations Oct/Dec et fêtes conservées."),
    ("paire-p03", "pun", False,
     "Il y a 10 types de personnes au monde : ceux qui comprennent le binaire et ceux qui ne le comprennent pas.",
     "Il y a à peu près 10 types de personnes au monde : ceux qui comprennent le binaire et ceux qui ne le comprennent pas.",
     3, 0,
     [("le '10' lu en base 2", ["10", "binaire", "base 2", "zero un", "deux en base dix"]),
      ("méta-récursivité de l'énoncé", ["meta", "lui-meme", "recursif", "auto"]),
      ("double lecture du nombre", ["double lecture", "deux sens", "ambigu", "dix"])],
     "L'approximation 'à peu près' force la lecture décimale triviale du '10' ; la phrase reste identique ailleurs."),
    ("paire-p04", "topical", True,
     "Un SQL entre dans un bar, voit deux tables et leur dit : 'SELECT * FROM...'",
     "Un SQL entre dans un bar, voit deux tables et demande laquelle est libre.",
     2, 0,
     [("double sens de 'tables'", ["table", "tables", "base de donnees"]),
      ("requête SELECT", ["select", "requete", "from", "interroge"]),
      ("personnification du langage", ["personnifi", "sql parle", "langage qui parle"])],
     "Retirer la punchline 'SELECT * FROM...' au profit d'une question littérale de bar ; la personne (SQL) et le décor (tables) restent."),
    ("paire-p05", "sociale", False,
     "Combien d'ingénieurs faut-il pour changer une ampoule ? Aucun, c'est un problème hardware.",
     "Combien d'ingénieurs faut-il pour changer une ampoule ? Un seul, muni d'un escabeau.",
     2, 0,
     [("inversion de responsabilité", ["responsabilite", "rejet", "renvoi", "pas au"]),
      ("clivage logiciel/matériel", ["hardware", "materiel", "software", "logiciel"]),
      ("réponse 'aucun' déflationnaire", ["aucun", "zero", "pas besoin d'ingenieur"])],
     "La réponse déflationnaire 'aucun + hardware' devient une réponse littérale ('un seul, escabeau') ; format question-réponse conservé."),
    ("paire-p06", "pun", False,
     "Je suis tombé amoureuse d'une fonction quadratique. Mais elle avait deux racines.",
     "Je suis tombé amoureuse d'une fonction quadratique. Mais elle avait deux zéros.",
     2, 0,
     [("double sens de 'racines'", ["racine", "root", "double sens"]),
      ("métaphore amoureuse mathématique", ["amoureus", "relation", "coeur", "sentiment"]),
      ("propriété de la quadratique", ["quadratique", "deux solutions", "deux racines", "polynome"])],
     "'racines' (double lecture sentimentale/math) remplacé par 'zéros' (terme technique neutre) ; la confession amoureuse est intacte."),
    ("paire-p07", "topical", True,
     "Un null et un undefined entrent dans un bar. Le barman dit 'On accepte pas les non-définis ici'.",
     "Un null et un undefined entrent dans un bar. Le barman leur demande ce qu'ils veulent boire.",
     2, 0,
     [("référence JavaScript", ["null", "undefined", "javascript", "js", "typage"]),
      ("double sens de 'non-définis'", ["non defini", "non-defini", "double sens", "exclu"]),
      ("mécanisme de blague de bar", ["bar", "barman", "refus", "entrent"])],
     "La réplique excluante ('on accepte pas les non-définis') devient un service ordinaire ; le couple null/undefined demeure."),
    ("paire-p08", "topical", True,
     "Je voulais te raconter une blague sur UDP... mais je sais si elle arrive.",
     "Je voulais te raconter une blague sur UDP... mais je ne sais pas si elle arrive.",
     3, 0,
     [("UDP sans garantie de livraison", ["udp", "best-effort", "best effort", "sans garantie", "livraison", "paquet"]),
      ("négation manquante = paquet perdu", ["negation", "ne...pas manquant", "paquet perdu", "omis"]),
      ("méta : la phrase incarne le protocole", ["meta", "la phrase elle-meme", "incarne"])],
     "Restaurer la négation grammaticale ('ne...pas') désamorce la chute : la phrase redevient littérale."),
    ("paire-p09", "pun", False,
     "Le père Noël a-t-il déjà eu un problème de pile ? Non, il a toujours des piles neuves.",
     "Le père Noël a-t-il déjà eu un problème de pile ? Non, ses lampes fonctionnent toujours bien.",
     1, 0,
     [("double sens de 'pile(s)'", ["pile", "piles", "double sens"]),
      ("réponse par l'objet électrique", ["batterie", "electrique", "neuve"]),
      ("cadre Noël", ["noel", "pere noel"])],
     "L'écho 'piles neuves' (reprise du mot de la question) remplacé par une réponse détournée ('lampes') ; le cadre questions-réponses reste."),
    ("paire-p10", "pun", False,
     "Pourquoi le café est-il si bon au travail ? Parce qu'il est fraîchement moulu par l'échéance.",
     "Pourquoi le café est-il si bon au travail ? Parce qu'il est fraîchement moulu chaque matin.",
     2, 0,
     [("métaphore échéance/mouture", ["echeance", "deadline", "moulu", "mouture"]),
      ("double sens de 'fraîchement moulu'", ["fraichement", "double sens", "moulu par"]),
      ("registre bureau", ["travail", "bureau", "cafe"])],
     "'par l'échéance' (agent incongru) devient 'chaque matin' (circonstance banale) ; question et registre identiques."),
    ("paire-p11", "pun", True,
     "J'ai essayé d'écrire une blague sur les coroutines, mais je n'arrive pas à la yield.",
     "J'ai essayé d'écrire une blague sur les coroutines, mais je n'arrive pas à la terminer.",
     2, 0,
     [("yield : mot-clé et céder/livrer", ["yield", "ceder", "mot-cle"]),
      ("référence coroutines", ["coroutine", "async", "generateur"]),
      ("double lecture du verbe", ["double lecture", "deux sens", "je n'arrive pas"])],
     "Le mot-clé 'yield' (double lecture) remplacé par le verbe banal 'terminer'."),
    ("paire-p12", "topical", True,
     "Docker, Kubernetes, Prometheus, Grafana. On m'a dit que c'était simple, alors je stack.",
     "Docker, Kubernetes, Prometheus, Grafana. On m'a dit que c'était simple, alors je m'y mets.",
     2, 0,
     [("double sens de 'stack'", ["stack", "empiler", "pile", "stacker"]),
      ("accumulation d'outils DevOps", ["docker", "kubernetes", "prometheus", "grafana", "outils"]),
      ("ironie de la promesse de simplicité", ["simple", "ironie", "promesse"])],
     "'je stack' (anglicisme-jeu) remplacé par 'je m'y mets' (locution neutre) ; l'énumération d'outils reste."),
    ("paire-p13", "pun", False,
     "Mon compilateur et moi on a une relation stable : il compile, je pleure.",
     "Mon compilateur et moi on a une relation stable : il compile, je vérifie.",
     2, 0,
     [("double sens de 'stable'", ["stable", "build", "relation"]),
      ("inversion émotionnelle (pleure alors que ça compile)", ["pleure", "inversion", "absurd", "alors que"]),
      ("personnification du compilateur", ["compilateur", "relation"])],
     "'je pleure' (réaction inversée, incongrue) devient 'je vérifie' (réaction attendue) ; 'relation stable' conservé."),
    ("paire-p14", "sociale", False,
     "J'ai un ami palindrome. On ne peut pas se différencier.",
     "J'ai un ami palindrome. C'est un mot comme un autre.",
     2, 0,
     [("propriété du palindrome appliquée à l'amitié", ["palindrome", "deux sens", "miroir", "endroit envers"]),
      ("absurdité de la catégorie (ami palindrome)", ["ami", "absurd", "personne", "mot"]),
      ("jeu sur la différenciation", ["differencier", "distinguer", "differents"])],
     "La seconde phrase cesse d'appliquer la propriété du palindrome à la relation ; elle ramène le palindrome à un fait lexical."),
    ("paire-p15", "topical", False,
     "Les regex sont comme des licornes : tout le monde en parle, personne les a vues.",
     "Les regex sont comme des licornes : on en voit surtout dans les livres.",
     2, 1,
     [("simile absurde (les regex se voient partout)", ["regex", "licorne", "personne ne", "jamais vue"]),
      ("véracité littérale fausse = incongruité", ["inversion", "faux", "exactement le contraire"]),
      ("mythe du métier", ["mythe", "legende", "metier", "rumeur"])],
     "La chute inversée ('personne les a vues') devient une vérité littérale sur les licornes ; le cadre de comparaison reste."),
    ("paire-p16", "sociale", False,
     "Un physicien, un biologiste et un chimiste voient 2 bâtiments. L'un entre, l'autre sort. 'Tiens, ils ont échangé.'",
     "Un physicien, un biologiste et un chimiste voient 2 bâtiments. L'un entre, l'autre sort. 'Tiens, ils se croisent.'",
     1, 0,
     [("trois corps de métier (format classique)", ["physicien", "biologiste", "chimiste", "trois"]),
      ("non-séquence finale ('ils ont échangé')", ["echange", "echangent", "non-sequence", "absurd"]),
      ("banalisation d'une observation triviale", ["observ", "tiens", "remarque"])],
     "'échangé' (verbe impossible pour des bâtiments) devient 'se croisent' (description littérale des faits)."),
    ("paire-p17", "topical", True,
     "Que dit un informaticien quand il s'ennuie ? 'printf(mot)\n'",
     "Que dit un informaticien quand il s'ennuie ? 'Il regarde par la fenêtre.'",
     1, 0,
     [("printf = dire/afficher", ["printf", "affiche", "print", "impression"]),
      ("réponse en code à une question de parole", ["code", "reponse en code", "dire"]),
      ("minimalisme absurde", ["minimal", "absurd", "laconique"])],
     "La réponse-code ('printf') remplacée par une réponse en parole ordinaire ; la question reste."),
    ("paire-p18", "topical", False,
     "Si Dieu existe, il est Objective-C : tout est message.",
     "Si Dieu existe, il est Objective-C : les conventions sont strictes.",
     2, 0,
     [("parallèle théologie/paradigme", ["dieu", "theolog", "religion", "objective-c"]),
      ("double sens de 'message' (passage de messages/révélation)", ["message", "passage de message", "revelation", "double sens"]),
      ("absorption : tout est X", ["tout est", "toute chose", "omni"])],
     "La proposition totalisante 'tout est message' (double lecture) remplacée par une remarque technique plate sur les conventions."),
    ("paire-p19", "pun", False,
     "Le temps est une illusion. Le décalage horaire, doublement.",
     "Le temps est une illusion. Le décalage horaire aussi.",
     2, 0,
     [("référence Hitchhiker's Guide", ["hitchhiker", "guide", "adam", "douglas"]),
      ("jeu sur l'adverbe 'doublement'", ["doublement", "double", "doubly"]),
      ("amplification en cascade", ["cascade", "encore plus", "amplifie"])],
     "L'adverbe inattendu 'doublement' (chute) devient l'extension plate 'aussi' ; la première phrase demeure."),
    ("paire-p20", "topical", False,
     "Un photon entre dans un bar et commande une bière. Le barman dit 'Pour vous, c'est gratuit, on vous voit pas partir'.",
     "Un photon entre dans un bar et commande une bière. Le barman lui sert la bière et lui souhaite la bonne soirée.",
     3, 0,
     [("physique du photon (vitesse c, temps propre)", ["photon", "lumiere", "vitesse", "temps propre", "relativite"]),
      ("incongruité : le barman applique la physique", ["barman", "gratuit", "voit pas partir", "incongrui"]),
      ("blague de bar détournée", ["bar", "biere", "commande"])],
     "La réplique physique du barman ('on vous voit pas partir') devient un service et une formule de politesse ordinaires."),
    ("paire-p21", "pun", False,
     "J'ai une blague sur les matrices, mais c'est hors de portée du public.",
     "J'ai une blague sur les matrices, mais elle est trop longue à raconter.",
     1, 0,
     [("double sens de 'portée' (range math/accessible)", ["portee", "range", "matrice", "math"]),
      ("méta-humour : retenir la blague", ["meta", "retient", "ne la raconte pas", "promesse"]),
      ("registre mathématique", ["matrice", "matrices", "algebre"])],
     "'hors de portée' (jeu mathématique) remplacé par le motif banal 'trop longue à raconter'."),
    ("paire-p22", "topical", False,
     "Un chat roux dans une salle de serveurs est dangereux : il pourrait activer l'incident majeur.",
     "Un chat roux dans une salle de serveurs est dangereux : il pourrait débrancher un câble.",
     1, 0,
     [("mème du chat roux = chaos", ["chat roux", "meme", "internet", "chaos"]),
      ("vocabulaire NOC/production", ["incident", "noc", "production", "severite"]),
      ("exagération absurde du risque", ["exagere", "absurd", "risque"])],
     "L'aboutissant spectaculaire ('activer l'incident majeur') devient un risque physique banal et plausible ('débrancher un câble')."),
    ("paire-p23", "sociale", False,
     "Pourquoi les plongeurs plongent-ils toujours en arrière et jamais en avant ? Parce que sinon ils tomberaient dans le bateau.",
     "Pourquoi les plongeurs plongent-ils toujours en arrière et jamais en avant ? Parce que c'est plus simple pour repartir du bateau.",
     3, 0,
     [("logique absurde de la chute", ["tomberaient", "bateau", "logique absurde", "inversion"]),
      ("format devinette", ["pourquoi", "devinette", "question"]),
      ("géométrie inversée (avant = bateau)", ["en avant", "arriere", "geometrie", "sens"])],
     "La chute géométriquement absurde ('tomberaient dans le bateau') devient une raison mécanique plausible ('repartir du bateau')."),
    ("paire-p24", "topical", False,
     "Le HTML n'est pas un langage de programmation. Et le plus dur, c'est de le dire à mon patron.",
     "Le HTML n'est pas un langage de programmation. Et le plus dur, c'est de le documenter proprement.",
     2, 0,
     [("débat tech canonique", ["html", "langage de programmation", "debat"]),
      ("tension hiérarchique (patron)", ["patron", "hierarch", "travail", "dire"]),
      ("déplacement : difficulté sociale d'un fait technique", ["le plus dur", "difficile de dire", "social"])],
     "La difficulté sociale ('dire à mon patron') devient une difficulté technique ('documenter proprement') ; l'affirmation technique reste."),
    ("paire-p25", "topical", False,
     "Si vous pensez que personne ne s'intéresse à votre vie, regardez vos logs Git.",
     "Si vous pensez que personne ne s'intéresse à votre vie, regardez vos notifications.",
     2, 0,
     [("métaphore de surveillance", ["surveill", "trace", "espion", "interesse"]),
      ("logs Git comme trace de vie", ["log", "git", "historique", "commits"]),
      ("réconfort ironique", ["personne ne", "ironie", "regarde"])],
     "'logs Git' (trace technique détournée en preuve d'intérêt) remplacé par 'notifications' (objet littéralement fait pour ça)."),
    ("paire-p26", "sociale", False,
     "Comment debug-on un avion ? On retire les composants un par un jusqu'à ce qu'il ne plante plus.",
     "Comment répare-t-on un avion ? On remplace les composants défectueux selon la procédure certifiée.",
     3, 0,
     [("transfert de la procédure de debug", ["debug", "bisection", "un par un", "procedure"]),
      ("double sens de 'plante' (crash)", ["plante", "crash", "double sens", "tomber en panne"]),
      ("danger absurde (avion de prod)", ["avion", "danger", "absurd"])],
     "'debug-on' devient 'répare-t-on', la bisection incongrue devient la maintenance certifiée ; le sujet (avion, composants) reste."),
    ("paire-p27", "pun", True,
     "Mieux vaut avoir un git pull que deux tu l'auras.",
     "Mieux vaut faire un git pull régulier que de tout merger d'un coup.",
     3, 0,
     [("paronomase du dicton", ["dicton", "proverbe", "mieux vaut", "tu l'auras"]),
      ("git pull substitué au mot attendu", ["git pull", "tirer", "fusion"]),
      ("double lecture de l'avoir", ["avoir", "double sens", "posseder"])],
     "La paronomase ('un git pull' / 'un qui tu l'auras') défusionnée en conseil technique plat."),
    ("paire-p28", "sociale", False,
     "Mon chat a appris Python. Maintenant il chasse les exceptions au lieu des souris.",
     "Mon chat a appris Python. Maintenant il marche sur le clavier au lieu de chasser les souris.",
     2, 0,
     [("chasse redirect vers les exceptions", ["chasse", "exception", "proie", "souris"]),
      ("anthropomorphisme technique", ["chat", "python", "appris", "anthropomorph"]),
      ("métabole 'au lieu de'", ["au lieu", "remplace", "desormais"])],
     "L'appariement absurde (chasser des exceptions) devient un comportement félin réel (marcher sur le clavier) ; la structure 'au lieu des souris' reste."),
    ("paire-p29", "sociale", False,
     "Les submodules Git, c'est comme les voisins : mieux vaut ne pas les déranger.",
     "Les submodules Git, c'est une fonctionnalité à manier avec précaution.",
     2, 0,
     [("métaphore sociale (voisins)", ["voisin", "voisins", "deranger", "social"]),
      ("folklore des submodules", ["submodule", "sous-module", "git"]),
      ("conseil de sagesse détourné", ["mieux vaut", "sagesse", "conseil"])],
     "La comparaison sociale ('comme les voisins') supprimée ; il reste une phrase technique prudente."),
    ("paire-p30", "pun", True,
     "Il était une fois un UTF-8 qui ne savait pas où était la fin. Il était perdu dans un BOM.",
     "Il était une fois un UTF-8 qui ne savait pas où était la fin. Il a trouvé la réponse dans la norme.",
     2, 0,
     [("conte détourné + jargon d'encodage", ["utf-8", "encodage", "bom", "conte", "il etait une fois"]),
      ("BOM : le délimiteur qui égare", ["bom", "marqueur", "octet", "debut"]),
      ("double lecture de 'la fin' (fin de chaîne/fin du conte)", ["fin", "chaine", "terminaison", "double lecture"])],
     "La chute technique ('perdu dans un BOM') remplacée par un dénouement plat ('la réponse dans la norme') ; le cadre du conte reste."),
]

FORMES = ["pun", "sociale", "topical"]
assert len(PAIRES_MINIMALES) == 30
assert all(p[1] in FORMES for p in PAIRES_MINIMALES)
n_par_forme = Counter(p[1] for p in PAIRES_MINIMALES)
n_ood = sum(1 for p in PAIRES_MINIMALES if p[2])
print(f"[paires] {len(PAIRES_MINIMALES)} paires, formes : {dict(n_par_forme)}, tranche OOD code-mixé : {n_ood}")
# Fidelite au corpus, verifiee AU RUN contre le corpus execute (cell[10]) :
# chaque texte humour doit etre le texte EXACT de POSITIVE_JOKES.
_corpus_par_id = {inst["id"]: inst["texte"] for inst in positive_instances}
for p in PAIRES_MINIMALES:
    ref = _corpus_par_id["joke-" + p[0].replace("paire-", "")]
    assert p[3] == ref, f"{p[0]} : texte humour != corpus\n  paire : {p[3]!r}\n  corpus: {ref!r}"
print("[paires] 30/30 textes humour verbatim du corpus (assert de fidélité passé)")

[paires] 30 paires, formes : {'sociale': 8, 'pun': 11, 'topical': 11}, tranche OOD code-mixé : 8
[paires] 30/30 textes humour verbatim du corpus (assert de fidélité passé)


In [15]:
# -*- coding: utf-8 -*-
# Controles de construction + GEL du split AVANT toute evaluation.
# Rien dans ce qui suit (seuils, split) ne sera retouche apres lecture du test.
import hashlib
import difflib

def lev_mots(a, b):
    """Distance de Levenshtein mot a mot (DP classique, sans dependance)."""
    wa, wb = a.split(), b.split()
    prev = list(range(len(wb) + 1))
    for i, xa in enumerate(wa, 1):
        cur = [i] + [0] * len(wb)
        for j, yb in enumerate(wb, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (xa != yb))
        prev = cur
    return prev[-1]

print("=== Contrôles de construction ===")
print(f"{'id':<12}{'forme':<9}{'mots H/U':<10}{'ratio':<7}{'lev mots':<9}{'sim chars':<10}ood")
ratios, sims = [], []
for (pid, forme, ood, th, tu, _, _, _, _) in PAIRES_MINIMALES:
    nh, nu = len(th.split()), len(tu.split())
    ratio = max(nh, nu) / max(1, min(nh, nu))
    lv = lev_mots(th, tu)
    sim = difflib.SequenceMatcher(None, th, tu).ratio()
    ratios.append(ratio)
    sims.append(sim)
    print(f"{pid:<12}{forme:<9}{nh:>3}/{nu:<4}{ratio:<7.2f}{lv:<9}{sim:<10.2f}{ood}")
print(f"\nratio longueur max = {max(ratios):.2f} (cible <= 1.35, consigne 'autant que possible')")
print(f"similarité car. min/max = {min(sims):.2f}/{max(sims):.2f} (des paires proches = voulu)")

# SEUILS DE VERDICT — declares et geles AVANT l'evaluation.
SEUILS = {
    "detection": {"generalise": 0.80, "par_forme_min": 0.70, "ne_generalise_pas": 0.60},
    "explication": {"noyau_generalise": 0.80, "completude_generalise": 0.60,
                    "noyau_ne_generalise_pas": 0.50},
    "appreciation": "INCONCLUSIVE_PAR_CONSTRUCTION (annotateur unique — règle de l'issue #14033)",
}
print("\n=== Seuils de verdict (gels avant mesure) ===")
print(json.dumps(SEUILS, indent=2, ensure_ascii=False))

# SPLIT GEL — stratifie par forme, moitie construction / moitie test
# (les effectifs impairs vont au CONSTRUCTION : pun 6/5, sociale 4/4, topical 6/5).
rng = random.Random(42)
construction_ids, test_ids = [], []
for f in FORMES:
    ids = sorted(p[0] for p in PAIRES_MINIMALES if p[1] == f)
    rng.shuffle(ids)
    k = (len(ids) + 1) // 2
    construction_ids.extend(ids[:k])
    test_ids.extend(ids[k:])
construction_ids.sort()
test_ids.sort()

CORPUS_VERSION = "gt24b-paires-v1"
corpus_blob = json.dumps(
    [[p[0], p[1], p[2], p[3], p[4], p[5], p[6]] for p in PAIRES_MINIMALES],
    ensure_ascii=False, sort_keys=True).encode("utf-8")
CORPUS_SHA = hashlib.sha256(corpus_blob).hexdigest()
SPLIT_SHA = hashlib.sha256(json.dumps(
    {"version": CORPUS_VERSION, "construction": construction_ids, "test": test_ids},
    sort_keys=True).encode("utf-8")).hexdigest()
print("\n=== GEL ===")
print(f"corpus {CORPUS_VERSION} sha256 = {CORPUS_SHA[:16]}...")
print(f"construction ({len(construction_ids)} paires) : {construction_ids}")
print(f"test ({len(test_ids)} paires) : {test_ids}")
print(f"split gelé sha256 = {SPLIT_SHA[:16]}...")
TEST_PAIRS = [p for p in PAIRES_MINIMALES if p[0] in set(test_ids)]
TEST_TEXTES = [(p[0], 1, p[3], p[5], p[1], p[2]) for p in TEST_PAIRS] + \
              [(p[0], 0, p[4], p[6], p[1], p[2]) for p in TEST_PAIRS]
CONSTR_PAIRS = [p for p in PAIRES_MINIMALES if p[0] in set(construction_ids)]
print(f"test : {len(TEST_PAIRS)} paires = {len(TEST_TEXTES)} textes ; construction : {len(CONSTR_PAIRS)} paires")

=== Contrôles de construction ===
id          forme    mots H/U  ratio  lev mots sim chars ood
paire-p01   sociale   13/15  1.15   3        0.89      False
paire-p02   pun       15/16  1.07   3        0.87      False
paire-p03   pun       22/25  1.14   3        0.95      False
paire-p04   topical   16/14  1.14   6        0.70      True
paire-p05   sociale   13/13  1.00   5        0.78      False
paire-p06   pun       12/12  1.00   1        0.95      False
paire-p07   topical   18/17  1.06   7        0.65      True
paire-p08   topical   14/16  1.14   2        0.95      True
paire-p09   pun       18/17  1.06   6        0.77      False
paire-p10   pun       16/16  1.00   2        0.90      False
paire-p11   pun       15/15  1.00   1        0.95      True
paire-p12   topical   13/14  1.08   2        0.94      True
paire-p13   pun       14/14  1.00   1        0.94      False
paire-p14   sociale   10/10  1.00   6        0.62      False
paire-p15   topical   16/14  1.14   8        0.69      F

### Lecture — 30 paires appariées, 3 formes, tranche OOD, split gelé avant mesure

- **11 pun / 8 sociale / 11 topical** — les trois formes exigées ; la pun et la référence culturelle dominent (le corpus source est un corpus de blagues d'informaticiens : biais déclaré, pas caché).
- **Tranche OOD = 8 paires code-mixées FR/EN** (`SELECT`, `null`/`undefined`, `UDP`, `yield`, `stack`, `printf`, `git pull`, `UTF-8`/`BOM`) — le code-mixing est exactement le régime que Horvitz et al. identifient comme difficile ; il est isolé pour être rapporté séparément.
- **Les éditions unfun conservent le décor** : ratios de longueur et similarités caractères mesurés ci-dessus ; la distance mot-à-mot reste faible — l'édit retire le mécanisme, pas le sujet.
- **Le split (16 construction / 14 test, stratifié par forme) et les seuils de verdict sont hachés avant la moindre évaluation** — aucune retouche possible après lecture du test.

In [16]:
# -*- coding: utf-8 -*-
# DEUX BASELINES sur EXACTEMENT le meme split test (aucune ne voit le test avant).
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

# --- Baseline 1 (controle) : le detecteur circulaire d'origine, reproduit ---
# On reconstruit les features DEPUIS LE LABEL, exactement comme la cellule de
# corpus initiale le faisait (cell[10]) : c'est le defaut que ce banc exhibe.
def features_circulaires(est_humour):
    return {"laugh": est_humour, "reframe": est_humour,
            "uptake": est_humour, "refus": False}

circ_pred = []
for (pid, cote, texte, note, forme, ood) in TEST_TEXTES:
    f = features_circulaires(cote == 1)
    pred = reframe_detector({"features": f})
    circ_pred.append(1 if pred == "humour_reussi" else 0)
circ_y = [t[1] for t in TEST_TEXTES]
f1_circ = f1_score(circ_y, circ_pred, average="macro")
print(f"[contrôle circulaire] macro-F1 = {f1_circ:.3f} — attendu 1.000 par construction :")
print("  les features dérivent du label, la règle inverse la dérivation. La métrique")
print("  est VIDE ; c'est la démonstration du défaut que les paires minimales corrigent.")

# --- Baseline 2 : classifieur lexical de surface, entraîné sur CONSTRUCTION seulement ---
constr_X = [p[3] for p in CONSTR_PAIRS] + [p[4] for p in CONSTR_PAIRS]
constr_y = [1] * len(CONSTR_PAIRS) + [0] * len(CONSTR_PAIRS)
lex = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), lowercase=True),
                    LogisticRegression(max_iter=1000))
lex.fit(constr_X, constr_y)
test_X = [t[2] for t in TEST_TEXTES]
lex_pred = list(lex.predict(test_X))
f1_lex = f1_score(circ_y, lex_pred, average="macro")
print(f"\n[baseline lexicale] macro-F1 test = {f1_lex:.3f} (TF-IDF 1-2 grammes +")
print("  régression logistique, entraînée sur les 16 paires de construction uniquement).")
print("  Sur le banc initial (positives tech vs négatives administratives) ce type de")
print("  classifieur réussissait par raccourci lexical ; ici les négatifs partagent le")
print("  lexique des positifs — le raccourci est fermé, la performance mesure la vraie tâche.")

[contrôle circulaire] macro-F1 = 1.000 — attendu 1.000 par construction :
  les features dérivent du label, la règle inverse la dérivation. La métrique
  est VIDE ; c'est la démonstration du défaut que les paires minimales corrigent.

[baseline lexicale] macro-F1 test = 0.535 (TF-IDF 1-2 grammes +
  régression logistique, entraînée sur les 16 paires de construction uniquement).
  Sur le banc initial (positives tech vs négatives administratives) ce type de
  classifieur réussissait par raccourci lexical ; ici les négatifs partagent le
  lexique des positifs — le raccourci est fermé, la performance mesure la vraie tâche.


### Lecture — la circularité rendue visible, le raccourci lexical fermé

La baseline circulaire rend **F1 = 1,000 par construction** : ce n'est pas une performance, c'est une tautologie mesurée. La baseline lexicale, elle, perd son raccourci : les contreparties « unfun » partagent le vocabulaire, le sujet et le registre des blagues — un n-gramme ne peut plus séparer « Oct 31 == Dec 25 » de « Oct 31 vient avant Dec 25 » sans comprendre lequel porte l'incongruité. Ces deux nombres encadrent l'évaluation LLM qui suit : **tout score LLM ≤ baseline lexicale ne démontre rien ; tout score ≈ 1,000 doit être regardé avec le même soupçon que la baseline circulaire.**

In [17]:
# -*- coding: utf-8 -*-
# Canal de jugement epingle : OpenRouter, modele ET version fixes (dispatch #14033).
def or_call(prompt, max_tokens=200, timeout=90):
    """Appel OpenRouter. Retourne (contenu, echo_modele, latence_ms, usage)."""
    body = json.dumps({
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
    }).encode("utf-8")
    req = urllib.request.Request(
        f"{OPENAI_COMPAT_URL}/chat/completions",
        data=body,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "HTTP-Referer": "https://github.com/jsboige/CoursIA",
            "X-Title": "CoursIA-GT24b-banc-humour-14033",
        },
    )
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=timeout) as r:
        resp = json.loads(r.read())
    dt = int((time.time() - t0) * 1000)
    content = resp["choices"][0]["message"]["content"].strip()
    return content, resp.get("model", "?"), dt, resp.get("usage", {})

_ping, _echo, _dt, _usage = or_call("Réponds uniquement : OK")
print(f"[canal] épinglé = {LLM_MODEL}")
print(f"[canal] écho provider = {_echo}  (résolution de version servie)")
print(f"[canal] latence = {_dt} ms ; usage = {_usage}")

[canal] épinglé = anthropic/claude-haiku-4.5
[canal] écho provider = anthropic/claude-haiku-4.5  (résolution de version servie)
[canal] latence = 860 ms ; usage = {'prompt_tokens': 14, 'completion_tokens': 4, 'total_tokens': 18, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 3.4e-05, 'upstream_inference_prompt_cost': 1.4e-05, 'upstream_inference_completions_cost': 2e-05}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}


In [18]:
# -*- coding: utf-8 -*-
# TACHE 1 — DETECTION. Binaire : le texte porte-t-il un mecanisme humoristique ?
# Les deux cotes de chaque paire test, ordre mélangé (anti-biais de position).
import unicodedata

def normaliser(s):
    s = unicodedata.normalize("NFD", s.lower())
    return "".join(c for c in s if not unicodedata.combining(c))

PROMPT_DETECTION = (
    "Tu es un détecteur d'humour. On te donne un texte court. Indique s'il contient "
    "un mécanisme humoristique volontaire (jeu de mots, incongruité, absurdité, chute). "
    "Réponds UNIQUEMENT par OUI ou NON.\n\nTexte : {texte}\n\nRéponse :")

rng_det = random.Random(7)
items_det = list(TEST_TEXTES)
rng_det.shuffle(items_det)
det_resultats = []
for (pid, cote, texte, note, forme, ood) in items_det:
    rep, echo, dt, _ = or_call(PROMPT_DETECTION.format(texte=texte), max_tokens=8)
    norm = normaliser(rep)
    pred = 1 if "oui" in norm else (0 if "non" in norm else -1)
    det_resultats.append({"pid": pid, "cote": cote, "pred": pred, "forme": forme,
                          "ood": ood, "brut": rep[:40], "ms": dt})
n_parse = sum(1 for r in det_resultats if r["pred"] >= 0)
print(f"[détection] {len(det_resultats)} appels, {n_parse} réponses OUI/NON parsées")

def macro_f1(rs):
    y = [r["cote"] for r in rs]
    p = [r["pred"] for r in rs]
    if -1 in p:
        p = [x if x >= 0 else 0 for x in p]  # non-parsé = NON
    return f1_score(y, p, average="macro")

f1_det = macro_f1(det_resultats)
print(f"\n=== DÉTECTION — macro-F1 (split test gelé) ===")
print(f"global                 : {f1_det:.3f}")
for f in FORMES:
    rs = [r for r in det_resultats if r["forme"] == f]
    print(f"forme {f:<9} ({len(rs):>2} txt) : {macro_f1(rs):.3f}")
rs_ood_det = [r for r in det_resultats if r["ood"]]
rs_fr_det = [r for r in det_resultats if not r["ood"]]
print(f"tranche OOD code-mixé  ({len(rs_ood_det):>2} txt) : {macro_f1(rs_ood_det):.3f}")
print(f"tranche FR pur         ({len(rs_fr_det):>2} txt) : {macro_f1(rs_fr_det):.3f}")
erreurs = [(r["pid"], r["cote"], r["pred"], r["forme"], r["ood"], r["brut"])
           for r in det_resultats if r["pred"] != r["cote"]]
print(f"\nerreurs ({len(erreurs)}) :")
for e in erreurs:
    print(f"  {e[0]} côté={e[1]} préd={e[2]} forme={e[3]} ood={e[4]} brut={e[5]!r}")

[détection] 28 appels, 28 réponses OUI/NON parsées

=== DÉTECTION — macro-F1 (split test gelé) ===
global                 : 0.509
forme pun       (10 txt) : 0.333
forme sociale   ( 8 txt) : 0.750
forme topical   (10 txt) : 0.333
tranche OOD code-mixé  ( 8 txt) : 0.333
tranche FR pur         (20 txt) : 0.560

erreurs (12) :
  paire-p25 côté=0 préd=1 forme=topical ood=False brut='OUI'
  paire-p12 côté=0 préd=1 forme=topical ood=True brut='OUI'
  paire-p30 côté=0 préd=1 forme=pun ood=True brut='OUI'
  paire-p27 côté=0 préd=1 forme=pun ood=True brut='OUI'
  paire-p28 côté=0 préd=1 forme=sociale ood=False brut='OUI'
  paire-p02 côté=0 préd=1 forme=pun ood=False brut='OUI'
  paire-p22 côté=0 préd=1 forme=topical ood=False brut='OUI'
  paire-p01 côté=1 préd=0 forme=sociale ood=False brut='NON'
  paire-p03 côté=0 préd=1 forme=pun ood=False brut='OUI'
  paire-p10 côté=0 préd=1 forme=pun ood=False brut='OUI'
  paire-p07 côté=0 préd=1 forme=topical ood=True brut='OUI'
  paire-p18 côté=0 préd=1 fo

In [19]:
# -*- coding: utf-8 -*-
# TACHE 2 — APPRECIATION. Ordinal 0-3 (échelle à la JEST, réserves documentées).
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

PROMPT_APPRECIATION = (
    "Tu es un évaluateur d'humour. Note à quel point ce texte est drôle pour un "
    "public adulte francophone, sur l'échelle ordinale :\n"
    "0 = pas drôle du tout\n1 = à peine drôle\n2 = moyennement drôle\n3 = très drôle\n"
    "Réponds UNIQUEMENT par le chiffre (0, 1, 2 ou 3).\n\nTexte : {texte}\n\nNote :")

rng_app = random.Random(11)
items_app = list(TEST_TEXTES)
rng_app.shuffle(items_app)
app_resultats = []
for (pid, cote, texte, note, forme, ood) in items_app:
    rep, echo, dt, _ = or_call(PROMPT_APPRECIATION.format(texte=texte), max_tokens=4)
    chiffres = [int(c) for c in rep if c in "0123"]
    pred = chiffres[0] if chiffres else None
    app_resultats.append({"pid": pid, "cote": cote, "annotateur": note,
                          "llm": pred, "forme": forme, "ood": ood})
valides = [r for r in app_resultats if r["llm"] is not None]
print(f"[appréciation] {len(valides)}/{len(app_resultats)} notes 0-3 parsées")

y_a = [r["annotateur"] for r in valides]
y_l = [r["llm"] for r in valides]
qwk = cohen_kappa_score(y_a, y_l, labels=[0, 1, 2, 3], weights="quadratic")
rho, pval = spearmanr(y_a, y_l)
print(f"\n=== APPRÉCIATION — accord LLM vs annotateur humain unique ===")
print(f"kappa quadratique pondéré = {qwk:.3f}")
print(f"Spearman rho = {rho:.3f} (p = {pval:.4f})")
for f in FORMES:
    rs = [r for r in valides if r["forme"] == f]
    if len(rs) >= 3:
        k = cohen_kappa_score([r["annotateur"] for r in rs], [r["llm"] for r in rs],
                              labels=[0, 1, 2, 3], weights="quadratic")
        print(f"forme {f:<9} : kappa_q = {k:.3f} (n={len(rs)})")
print(f"\ndistribution annotateur : {dict(Counter(y_a))}")
print(f"distribution LLM         : {dict(Counter(y_l))}")
print("\nVERDICT APPRÉCIATION = INCONCLUSIVE_PAR_CONSTRUCTION : un seul annotateur")
print("humain (l'auteur du corpus) — la règle de l'issue #14033 impose ce verdict ;")
print("l'accord rapporté ci-dessus est une donnée, pas une généralisation. L'échelle")
print("JEST (Toplyn & Amir 2026) est elle-même limitée aux textes conversationnels")
print("courts et aux adultes américains : réserve doublée, non levée ici.")

[appréciation] 28/28 notes 0-3 parsées

=== APPRÉCIATION — accord LLM vs annotateur humain unique ===
kappa quadratique pondéré = 0.355
Spearman rho = 0.493 (p = 0.0076)
forme pun       : kappa_q = 0.238 (n=10)
forme sociale   : kappa_q = 0.564 (n=8)
forme topical   : kappa_q = 0.333 (n=10)

distribution annotateur : {0: 14, 3: 4, 2: 9, 1: 1}
distribution LLM         : {1: 15, 2: 12, 0: 1}

VERDICT APPRÉCIATION = INCONCLUSIVE_PAR_CONSTRUCTION : un seul annotateur
humain (l'auteur du corpus) — la règle de l'issue #14033 impose ce verdict ;
l'accord rapporté ci-dessus est une donnée, pas une généralisation. L'échelle
JEST (Toplyn & Amir 2026) est elle-même limitée aux textes conversationnels
courts et aux adultes américains : réserve doublée, non levée ici.


In [20]:
# -*- coding: utf-8 -*-
# TACHE 3 — EXPLICATION. Grille humaine explicite : exactitude (noyau) + complétude.
PROMPT_EXPLICATION = (
    "Explique en 1 à 3 phrases le mécanisme humoristique de ce texte : quelle "
    "incongruité, quelle(s) référence(s), quelle norme il joue. Si le texte ne "
    "présente pas de mécanisme humoristique clair, écris uniquement : "
    "PAS_DE_MECANISME\n\nTexte : {texte}\n\nExplication :")

def score_grille(explication, mecanisme):
    """Grille explicite : normalise (casse+accents), cherche les mots-clés.
    Rend (noyau_trouvé, n_éléments_trouvés, n_éléments)."""
    norm = normaliser(explication)
    trouves = 0
    noyau = False
    for i, (element, mots) in enumerate(mecanisme):
        hit = any(normaliser(m) in norm for m in mots)
        trouves += hit
        if i == 0 and hit:
            noyau = True
    return noyau, trouves, len(mecanisme)

rng_exp = random.Random(23)
items_expl = sorted(TEST_PAIRS, key=lambda p: p[0])
rng_exp.shuffle(items_expl)
expl_resultats = []
for p in items_expl:
    pid, forme, ood, th, tu, nh, nu, meca, jedit = p
    rep, echo, dt, _ = or_call(PROMPT_EXPLICATION.format(texte=th), max_tokens=250)
    noyau, nt, ntot = score_grille(rep, meca)
    expl_resultats.append({"pid": pid, "forme": forme, "ood": ood,
                           "noyau": noyau, "couverture": nt / ntot,
                           "pas_de_mecanisme": "PAS_DE_MECANISME" in rep.upper(),
                           "texte": rep})
print(f"[explication] {len(expl_resultats)} blagues du split test expliquées")

acc_noyau = sum(r["noyau"] for r in expl_resultats) / len(expl_resultats)
completude = sum(r["couverture"] for r in expl_resultats) / len(expl_resultats)
print(f"\n=== EXPLICATION — grille humaine explicite ===")
print(f"exactitude (élément NOYAU nommé) = {acc_noyau:.3f}")
print(f"complétude (moyenne des couvertures) = {completude:.3f}")
for f in FORMES:
    rs = [r for r in expl_resultats if r["forme"] == f]
    if rs:
        print(f"forme {f:<9} : noyau {sum(r['noyau'] for r in rs)/len(rs):.3f} "
              f"complétude {sum(r['couverture'] for r in rs)/len(rs):.3f} (n={len(rs)})")
rs_ood_expl = [r for r in expl_resultats if r["ood"]]
if rs_ood_expl:
    print(f"tranche OOD          : noyau {sum(r['noyau'] for r in rs_ood_expl)/len(rs_ood_expl):.3f} "
          f"complétude {sum(r['couverture'] for r in rs_ood_expl)/len(rs_ood_expl):.3f} (n={len(rs_ood_expl)})")
print("\nexemples (2 premiers) :")
for r in expl_resultats[:2]:
    print(f"  {r['pid']} [noyau={r['noyau']} couv={r['couverture']:.2f}] {r['texte'][:180]}")
print("\nréserve de méthode : la grille est un appariement de mots-clés (normalisés),")
print("pas une lecture sémantique — elle sous-compte les paraphrases correctes et")
print("sur-compte les mentions de passage. Règle déclarée avant mesure, appliquée")
print("mécaniquement : falsifiable, jamais retouchée après lecture du test.")

[explication] 14 blagues du split test expliquées

=== EXPLICATION — grille humaine explicite ===
exactitude (élément NOYAU nommé) = 0.857
complétude (moyenne des couvertures) = 0.762
forme pun       : noyau 1.000 complétude 0.867 (n=5)
forme sociale   : noyau 0.500 complétude 0.500 (n=4)
forme topical   : noyau 1.000 complétude 0.867 (n=5)
tranche OOD          : noyau 1.000 complétude 0.833 (n=4)

exemples (2 premiers) :
  paire-p22 [noyau=True couv=1.00] Le texte joue sur une homophonie entre "incident majeur" (problème informatique grave) et "incident majeur" prononcé comme "in-chat-dent majeur" (jeu de mots avec "chat"). L'incong
  paire-p27 [noyau=True couv=1.00] Ce texte joue sur l'incongruité entre un proverbe français classique ("Mieux vaut avoir un tiens que deux tu l'auras") et le jargon informatique en remplaçant "un tiens" par "un gi

réserve de méthode : la grille est un appariement de mots-clés (normalisés),
pas une lecture sémantique — elle sous-compte les paraphrases co

In [21]:
# -*- coding: utf-8 -*-
# CONTROLES D'HONNETETE + VERDICTS FERMES (seuils gelés en amont).
import hashlib as _hl

# Near-duplicate construction <-> test (hors paire elle-même : la ressemblance
# intra-paire est le DESSEIN du banc ; on vérifie qu'aucune paire de construction
# ne pré-crée une paire de test par quasi-doublon).
max_ratio, argmax = 0.0, None
for pc in CONSTR_PAIRS:
    for pt in TEST_PAIRS:
        for tc in (pc[3], pc[4]):
            for tt in (pt[3], pt[4]):
                r = difflib.SequenceMatcher(None, tc, tt).ratio()
                if r > max_ratio:
                    max_ratio, argmax = r, (pc[0], pt[0])
print(f"[dédup] similarité max construction<->test = {max_ratio:.2f} {argmax}")
hashes = [_hl.sha256((p[3] + p[4]).encode("utf-8")).hexdigest() for p in PAIRES_MINIMALES]
print(f"[dédup] doublons exacts intra-corpus : {len(hashes) - len(set(hashes))}")

print("\n[provenance] 30 textes humores + 30 contreparties unfun : contenu propre")
print("  écrit pour ce dépôt (lignée #12785, POSITIVE_JOKES cell[10]) ; les éditions")
print("  unfun sont inédites, créées pour #14033. Aucune donnée Reddit/Kaggle (les")
print("  verdicts REFERENCE_ONLY de l'issue restent en vigueur). Droits établis.")
print("[contenu] aucune tranche offensive dans les 30 paires ; la catégorie")
print("  offensif_compris_non_partage du banc initial reste dans sa tranche séparée,")
print("  jamais assimilée à du non-humour.")
print(f"[version] {CORPUS_VERSION} sha256={CORPUS_SHA[:16]}... split sha256={SPLIT_SHA[:16]}...")
print("[contamination] dépôt public sur GitHub : tout modèle entraîné sur le web a pu")
print("  voir les textes HUMORES ; les contreparties unfun sont inédites et font écran.")

print("\n=== VERDICTS FERMES (règle #14033 : jamais 'promising') ===")
s = SEUILS["detection"]
f1_par_forme = {f: macro_f1([r for r in det_resultats if r["forme"] == f]) for f in FORMES}
f1_ood = macro_f1(rs_ood_det)
if (f1_det >= s["generalise"]
        and all(v >= s["par_forme_min"] for v in f1_par_forme.values())
        and f1_ood >= s["par_forme_min"]):
    v_det = "GENERALISE"
elif f1_det < s["ne_generalise_pas"]:
    v_det = "NE_GENERALISE_PAS"
else:
    v_det = "INCONCLUSIVE"
print(f"DETECTION     : {v_det} (global {f1_det:.3f}, formes {f1_par_forme}, OOD {f1_ood:.3f})")
print(f"APPRECIATION  : INCONCLUSIVE_PAR_CONSTRUCTION (annotateur unique, kappa_q = {qwk:.3f})")
se = SEUILS["explication"]
if acc_noyau >= se["noyau_generalise"] and completude >= se["completude_generalise"]:
    v_exp = "GENERALISE"
elif acc_noyau < se["noyau_ne_generalise_pas"]:
    v_exp = "NE_GENERALISE_PAS"
else:
    v_exp = "INCONCLUSIVE"
print(f"EXPLICATION   : {v_exp} (noyau {acc_noyau:.3f}, complétude {completude:.3f})")
print(f"résiduel explicite : baselines sur 28 textes de test / 14 explications ;")
print(f"corpus mono-culture (blagues d'informaticiens FR, auteur = unique annotateur)")

[dédup] similarité max construction<->test = 0.55 ('paire-p04', 'paire-p07')
[dédup] doublons exacts intra-corpus : 0

[provenance] 30 textes humores + 30 contreparties unfun : contenu propre
  écrit pour ce dépôt (lignée #12785, POSITIVE_JOKES cell[10]) ; les éditions
  unfun sont inédites, créées pour #14033. Aucune donnée Reddit/Kaggle (les
  verdicts REFERENCE_ONLY de l'issue restent en vigueur). Droits établis.
[contenu] aucune tranche offensive dans les 30 paires ; la catégorie
  offensif_compris_non_partage du banc initial reste dans sa tranche séparée,
  jamais assimilée à du non-humour.
[version] gt24b-paires-v1 sha256=116425d0222c5cc1... split sha256=2aab8cbb7e9982e5...
[contamination] dépôt public sur GitHub : tout modèle entraîné sur le web a pu
  voir les textes HUMORES ; les contreparties unfun sont inédites et font écran.

=== VERDICTS FERMES (règle #14033 : jamais 'promising') ===
DETECTION     : NE_GENERALISE_PAS (global 0.509, formes {'pun': 0.3333333333333333, 'socia

### Conclusion — un banc falsifiable, trois verdicts séparés

Le détecteur du banc initial ne démontrait rien : ses features dérivaient du label (F1 = 1,000 tautologique, reproduit ci-dessus comme **contrôle**). Sur des **paires minimales** dont les négatifs partagent lexique, sujet et registre des positifs, la baseline lexicale perd son raccourci et le score LLM redevient une **vraie mesure** — séparée en trois questions indépendantes :

1. **détection** — macro-F1 global, par forme (pun / sociale / topical) et par tranche (code-mixé / FR pur), avec les erreurs listées une à une ;
2. **appréciation** — κ quadratique pondéré et ρ de Spearman contre l'annotateur humain unique, verdict **INCONCLUSIVE par construction** (règle de l'issue : un seul annotateur ne généralise pas une échelle de goût) ;
3. **explication** — grille humaine explicite (élément noyau + éléments secondaires), exactitude et complétude rapportées séparément, réserve de méthode déclarée (appariement de mots-clés, pas lecture sémantique).

Les contrôles d'honnêteté (quasi-doublons construction↔test, hachage du corpus et du split gelé, provenance et droits, tranche offensive isolée, contamination déclarée) sont mesurés, pas affirmés. Les verdicts fermes — `GENERALISE` / `NE_GENERALISE_PAS` / `INCONCLUSIVE` — sont calculés par les seuils gelés **avant** mesure ; « promising » n'existe pas ici.